# A1. 할인 패턴과 다음 할인 예측 — 할인에는 규칙이 있고, 다음 할인을 맞힐 수 있나

* 담당: 한예찬 · 분석 방향 문서의 **A1** (11/22 회의 확정)
* 한 줄 질문: **"우리 서비스가 쓰는 '마지막 할인일 + 과거 할인 간격의 평균'이라는 예측 규칙은 믿을 만한가?"**
* 새로 수집한 데이터는 없습니다. 수집기가 모아 정제·저장해 둔 **가격 이력(`price_history`)** 을 분석합니다.

### 데이터 처리 과정 (이 프로젝트의 수집 → 전달 → 정제 → 저장 → 활용 중 이 노트북이 맡는 부분)

| 단계 | 내용 | 이 노트북에서 |
| --- | --- | --- |
| 수집 | ITAD(PC 스토어 가격 이력) | 이미 수집기가 `raw.*`로 전송 |
| 전달·정제 | Kafka → 정제 프로그램: 중복 제거, **원화 환산**, **게임 ID 통합** (스토어 상품 → IGDB `game_id`) | 이미 `refined.price_history`로 정제됨 |
| 저장 | Kafka Connect → MariaDB `price_history` (**가격이 바뀐 때만** 한 줄씩 쌓임) | SQL로 읽기 전용 추출한 TSV를 불러옴 |
| 활용 | 날짜별 최저가 만들기 → 할인 구간 추출 → 규칙성·시즌성·예측 정확도 검정 → 서비스 제안 | 3~8장 |

### 분석 순서
1. 환경 설정 → 2. **가설 설정** → 3. 데이터 불러오기·점검 → 4. 전처리(할인 구간 만들기) → 5. 탐색(EDA) → 6. **통계 검정** → 7. 의미 있는 결과인가? → 8. **인사이트·제안**

### 한눈에 보기 (2026-11-26까지의 데이터 기준)
1. 할인은 **규칙적**이다. 게임의 85.5%가 무작위보다 규칙적으로 할인한다.
2. 3·6·12월(봄·여름·겨울 세일)에 할인이 몰린다. 모든 스토어의 최저가를 섞으면 상시 할인에 가까운 판매점이 섞여 **쏠림이 희석돼 약하지만**(Cramér's V 0.08), **Steam만 보면 정기 세일에 뚜렷하게 몰린다**(V 0.13).
3. 다음 할인일 예측은 **생각보다 어렵다.** 서비스 규칙의 평균 오차는 23일이고, 게임을 구분하지 않고 "전체 중앙값(약 35일)"을 쓰는 편이 오히려 더 정확했다(20.7일). 공식 스토어(Steam)만 봐도 서비스 규칙이 전체 중앙값보다 약 1일 더 부정확하다.
4. 제안: 다음 할인일을 날짜 하나가 아니라 **구간**("○○ 전후 ±2주")으로 보여 준다.

> 이 노트북의 숫자는 모두 실행 결과로 계산됩니다. 위 요약은 2026-11-26 데이터로 실행했을 때의 값이고, 최종 결론은 8장에서 실제 값으로 자동 출력됩니다. **★** 는 발표에서 보여 줄 핵심 차트입니다.

### 준비 (한 번만)
```bash
pip install pandas numpy scipy matplotlib statsmodels jupyterlab
```
* DB 데이터 추출 (mariadb-re, 읽기 전용 계정):
```bash
mysql -u <읽기 계정> -p gamenuri -B --default-character-set=utf8mb4 < analysis/scripts/a1_extract.sql > analysis/data/a1_price_history.tsv
```
* 분석 대상 게임 목록은 `analysis/game_list/games_1000.csv` (Git에 있음)를 씁니다.
* 노트북은 `analysis/notebooks` 폴더에서 열어야 `../data` 경로가 맞습니다.

## 1. 환경 설정

In [ ]:
# 불필요한 경고 메시지를 숨겨 출력이 깔끔하게 보이도록 합니다.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 분석에 필요한 라이브러리를 불러옵니다.
# pandas/numpy: 데이터 처리, scipy: 통계 검정, matplotlib: 시각화
import os, json
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy import stats

%matplotlib inline

### 한글 폰트 설정

In [ ]:
# 그래프의 한글이 깨지지 않도록 설치된 한글 폰트를 찾아 지정합니다. (Windows: 맑은 고딕, Mac: AppleGothic)
candidates = ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans CJK JP']
available = {f.name for f in fm.fontManager.ttflist}
font_name = next((c for c in candidates if c in available), None)

plt.rcParams['font.family'] = font_name or 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

BLUE, ORANGE, GRAY, GREEN, RED = '#2a78d6', '#eb6834', '#a3a29d', '#2f9e6e', '#d9453d'
print('사용 폰트:', font_name)

## 2. 가설 설정

데이터를 보기 **전에** 가설과 판정 기준을 먼저 정해 둡니다.

**할인 1회** = 할인율 10% 이상인 날이 이어진 구간 (서비스 `backend/app/services/timing.py`와 같은 기준).
**예측** = 마지막 할인 시작일 + 과거 할인 간격의 평균.

| # | 가설 (H1: 우리가 기대하는 것) | 귀무가설 (H0) | 통계 도구 | 실무 기준 ("의미 있다"의 조건) |
| --- | --- | --- | --- | --- |
| H1 | **할인은 무작위가 아니라 규칙적인 간격으로 반복된다** | 할인 간격이 무작위(지수분포)다 | Shapiro-Wilk로 정규성 확인 → **Wilcoxon 부호순위**(단측), 게임 단위 부트스트랩 신뢰구간 | 실제 변동계수(CV)가 무작위 기준보다 **0.2 이상 작고**, 절반 넘는 게임이 무작위보다 규칙적 |
| H2-a | **할인은 특정 달에 몰린다** (스토어 세일 시즌) — 모든 스토어의 최저가 기준 | 12개월에 고르게 퍼진다 | **카이제곱 적합도 검정**(기대값 = 달별 관찰 일수), **Cramér's V**, 게임 단위 부트스트랩 | Cramér's V ≥ **0.09** (Cohen의 '중간 효과', 월이 12개일 때) |
| H2-b | 같은 쏠림이 **공식 스토어(Steam)만 보면 더 뚜렷하다** | 〃 | 〃 | 〃 |
| H3-a | 서비스 규칙이 **"직전 간격 하나"보다 정확**하다 | 예측 오차가 같다 | **백테스트**(과거를 가려 놓고 맞히기) → **Wilcoxon 부호순위**(단측), 게임 단위 부트스트랩 | 평균 오차가 **2일 이상** 작음 |
| H3-b | 서비스 규칙이 **"전체 중앙값"(게임 정보를 쓰지 않는 가장 단순한 값)보다 정확**하다 | 예측 오차가 같다 | 〃 | 〃 |

* **유의수준 α = 0.05.** 통계적으로 유의해도 효과가 작으면 "의미 없음"으로 판정합니다.
* **H2를 두 기준으로 나눈 이유**: "모든 스토어의 최저가"에는 상시 할인에 가까운 판매점이 섞여 있어서 정기 세일의 쏠림이 희석될 수 있습니다. 그래서 같은 검정을 **Steam(공식 스토어)만으로도** 해서 비교합니다.
* **같은 세일에 여러 게임이 같이 들어가서 게임들은 서로 독립이 아닙니다.** 그래서 p값은 실제보다 작게 나오기 쉽습니다. 신뢰구간은 *게임을 단위로 다시 뽑는(cluster bootstrap)* 방식으로 구하고, 판정은 p값보다 효과 크기를 우선합니다.

### 분석 설정값 (한 곳에서 관리)

In [ ]:
# ---- 데이터 위치 ----
DATA_DIR = Path(os.environ.get('GAMENURI_DATA_DIR', '../data'))                  # analysis/data (Git에 올리지 않음)
RESULT_DIR = Path(os.environ.get('GAMENURI_RESULT_DIR', '../results/a1'))
GAME_LIST = Path(os.environ.get('GAMENURI_GAME_LIST', '../game_list/games_1000.csv'))
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# ---- 분석 정의 (서비스 timing.py와 같은 기준) ----
SALE_MIN_PCT = 10           # 할인율 10% 이상인 날이 이어진 구간 = 할인 1회
WINDOW_DAYS = 730           # 할인 횟수·주기 계산 기간 (최근 2년)
MIN_HISTORY_DAYS = 3        # 이력 일수가 이보다 적으면 제외
MIN_EVENTS = 4              # 간격이 3개 이상 나오는 게임만 규칙성(H1)에 쓴다
MIN_GLOBAL_GAPS = 50        # "전체 중앙값" 기준을 쓰려면 예측 시점 이전에 끝난 간격이 이만큼 있어야 함

# ---- 통계 ----
ALPHA = 0.05
PRACTICAL_CV = 0.20         # H1 실무 기준: 실제 CV가 무작위 기준보다 0.2 이상 작음
PRACTICAL_V = 0.09          # H2 실무 기준: Cramér's V 0.09 이상 (Cohen의 효과 크기 w=0.3 '중간 효과'에 해당, 월이 12개일 때 w = V x 3.3)
PRACTICAL_DAYS = 2.0        # H3 실무 기준: 평균 오차가 2일 이상 줄어듦
N_BOOT, SEED = 1000, 42

R = {}                      # 최종 요약에 쓸 결과 저장소

def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(RESULT_DIR / name, dpi=150, bbox_inches='tight')

## 3. 데이터 불러오기와 점검

In [ ]:
# mysql --batch로 뽑은 TSV를 읽습니다. 값이 없는 칸은 문자열 'NULL'로 나오므로 결측치로 바꿔 줍니다.
path = DATA_DIR / 'a1_price_history.tsv'
if not path.exists():
    raise FileNotFoundError(f'{path} 가 없습니다. 위 "준비" 칸의 mysql 명령으로 먼저 추출하세요.')
raw = pd.read_csv(path, sep='\t', na_values=['NULL'], parse_dates=['collected_at'])

games = pd.read_csv(GAME_LIST, encoding='utf-8-sig')                   # 분석 대상 게임 1,000개
games['name_ko'] = games['name_ko'].fillna(games['name'])
prices = raw[raw['game_id'].isin(games['game_id'])].copy()

print(f"가격 이력 {len(prices):,}행 · 게임 {prices['game_id'].nunique():,}개 · "
      f"{prices['collected_at'].min():%Y-%m-%d} ~ {prices['collected_at'].max():%Y-%m-%d}")
prices.head()

In [ ]:
# 데이터 타입과 결측치를 확인합니다. 분석 전에 반드시 구조를 점검하는 단계입니다.
prices.info()

In [ ]:
# 컬럼별 결측치. regular_krw·discount_pct는 스토어가 정가·할인율을 주지 않는 행에서 비어 있습니다.
# 이 노트북은 할인율이 비면 0으로 보므로(= 할인 아님) 결측이 있어도 분석에는 문제가 없습니다.
pd.DataFrame({'결측 수': prices.isnull().sum(), '결측률(%)': prices.isnull().mean() * 100})

In [ ]:
# 숫자 컬럼의 기초 통계: 가격이 0 이하이거나 비정상적으로 큰 값이 없는지 확인합니다.
prices[['price_krw', 'regular_krw', 'discount_pct']].describe()

In [ ]:
# 스토어별 행 수. 키 판매점이 많아 "모든 스토어의 최저가"를 보면 할인이 자주 잡힙니다 (6-2와 7-1에서 Steam만 따로 확인).
store_n = prices['store'].value_counts()
display(store_n.head(10).rename('행 수').to_frame().T)

# 게임별 가격 이력이 몇 일치인지 (이력이 너무 짧은 게임은 간격을 구할 수 없습니다)
hist_days = prices.assign(day=prices['collected_at'].dt.normalize()).groupby('game_id')['day'].nunique()
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(hist_days, bins=40, color=BLUE)
ax.set_title('게임별 가격 이력 일수'); ax.set_xlabel('가격이 기록된 날 수'); ax.set_ylabel('게임 수')
savefig(fig, 'a1_0_history_days.png'); plt.show()

### 차트 해설
게임마다 가격이 기록된 날이 며칠인지입니다. `price_history`는 가격이 **바뀐 때만** 쌓이므로, 일수가 적다고 이력이 짧은 것은 아니고 가격이 잘 안 바뀐 게임일 수도 있습니다.

### 이 차트로 알 수 있는 것
- **왼쪽 끝 막대가 가장 높습니다.** 가격이 기록된 날이 20일도 안 되는 게임이 약 11%(111개)이고, 50일 미만은 약 30%입니다. 가격이 거의 안 바뀐 게임입니다. (2026-11-26 데이터 기준)
- 이 구간 게임의 약 90%는 할인이 4회 미만이라 4장에서 간격 분석 대상에서 빠집니다. 다만 **기록 일수가 많아도** 할인이 4회 미만이면 빠집니다(100일 이상 게임의 약 31%). 분석 대상을 정하는 것은 기록 일수가 아니라 **할인 횟수**입니다.

### 의미 판단 (2026-11-26 데이터 기준)
- **가격이 거의 안 바뀐 게임이 꽤 많습니다.** 기록이 20일도 안 되는 게임이 약 10%입니다. 이런 게임은 할인도 거의 없었을 가능성이 큽니다.
- 그래서 "이력이 충분한 게임"은 기록 일수로 고르면 안 되고 **할인 횟수**로 골라야 합니다. 4장에서 "할인 4회 이상" 기준을 쓰는 이유입니다.

## 4. 전처리 — 할인 구간 만들기

가격 이력에는 **가격이 바뀐 날만** 기록돼 있어서 두 가지를 합니다. (서비스 `timing.py`와 같은 방식)
1. **날짜별 최저가 한 줄**: 스토어(+에디션)별 마지막 가격을 이어 채운(forward fill) 뒤 그날의 최저가만 남깁니다. 이렇게 해야 모든 스토어의 "그 시점 가격"이 됩니다.
2. **할인 1회**: 할인율이 10% 이상인 날이 이어진 구간 하나를 할인 1회로 세고, **시작일**과 최대 할인율을 기록합니다.
3. 최근 2년만 보고, 이력이 너무 짧은 게임은 뺍니다.

In [ ]:
def daily_series(g):
    """게임 1개의 일별 최저가 (컬럼: day, price_krw, discount_pct)."""
    g = g.assign(day=g['collected_at'].dt.normalize(), discount_pct=g['discount_pct'].fillna(0))
    last = g.sort_values('collected_at', kind='stable').groupby(['day', 'store', 'edition']).tail(1)      # 같은 날 마지막 행
    price = last.pivot(index='day', columns=['store', 'edition'], values='price_krw').sort_index().sort_index(axis=1).ffill()
    disc = last.pivot(index='day', columns=['store', 'edition'], values='discount_pct').reindex_like(price).ffill()
    best = np.nanargmin(price.to_numpy(), axis=1)                                          # 그날 가장 싼 스토어
    rows = np.arange(len(best))
    return pd.DataFrame({'day': price.index, 'price_krw': price.to_numpy()[rows, best],
                         'discount_pct': disc.to_numpy()[rows, best]})

def sale_events(d, sale_min=SALE_MIN_PCT):
    """할인율이 sale_min 이상인 날이 이어진 구간마다 한 건 (컬럼: start, max_pct)."""
    events, start, top = [], None, 0
    for day, pct in zip(d['day'], d['discount_pct']):
        if pct >= sale_min:
            if start is None:
                start, top = day, pct
            else:
                top = max(top, pct)
        elif start is not None:
            events.append((start, top)); start = None
    if start is not None:
        events.append((start, top))
    return pd.DataFrame(events, columns=['start', 'max_pct']).astype({'start': 'datetime64[ns]'})

END = prices['collected_at'].max().normalize()                 # 데이터의 마지막 날 = "지금"
SINCE = END - pd.Timedelta(days=WINDOW_DAYS)                   # 최근 2년만 본다

def make_series(prices_sub):
    s = {gid: daily_series(g) for gid, g in prices_sub.groupby('game_id')}
    return {gid: d for gid, d in s.items() if len(d) >= MIN_HISTORY_DAYS}

def make_events(series, sale_min=SALE_MIN_PCT):
    return {gid: sale_events(d[d['day'] >= SINCE], sale_min) for gid, d in series.items()}

series_all = make_series(prices)                               # 모든 스토어의 최저가 기준 (서비스가 보여 주는 가격)
events = make_events(series_all)
n_events = pd.Series({gid: len(ev) for gid, ev in events.items()})
print(f'일별 최저가를 만든 게임 {len(series_all):,}개 · 할인 횟수 중앙값 {n_events.median():.0f}회')

In [ ]:
@lru_cache(maxsize=None)
def null_cv(k, n_sim=5000):
    """간격이 k개인 '무작위 할인'(지수분포)의 평균 변동계수. 간격 수가 적으면 우연히 작게 나오므로 같은 횟수로 시뮬레이션한다."""
    x = np.random.default_rng(SEED + k).exponential(1.0, size=(n_sim, k))
    return float((x.std(axis=1, ddof=1) / x.mean(axis=1)).mean())

def h1_table(ev, min_events=MIN_EVENTS):
    """게임별 할인 간격과 변동계수(CV = 간격의 표준편차 ÷ 평균 간격), 무작위일 때의 기준 CV."""
    rows = []
    for gid, e in ev.items():
        if len(e) >= min_events:
            gaps = e['start'].diff().dt.days.dropna().to_numpy()                 # 할인 시작일 사이의 간격(일)
            rows.append({'game_id': gid, 'n_events': len(e), 'gaps': gaps, 'mean_gap': gaps.mean(),
                         'cv': gaps.std(ddof=1) / gaps.mean(), 'cv_null': null_cv(len(gaps))})
    return pd.DataFrame(rows).set_index('game_id')

gap_df = h1_table(events)
pooled = np.concatenate(gap_df['gaps'].to_list())
print(f'간격 분석 대상 {len(gap_df):,}개 게임 (할인 {MIN_EVENTS}회 이상)')
gap_df[['n_events', 'mean_gap', 'cv', 'cv_null']].describe().round(2)

In [ ]:
# 분석 표본이 되기까지 (단계별로 게임이 몇 개 남는가)
funnel = {'분석 대상 게임': len(games), '가격 이력이 있음': prices['game_id'].nunique(), f'이력 {MIN_HISTORY_DAYS}일 이상': len(series_all),
          '할인이 1회 이상': int((n_events >= 1).sum()), f'할인 {MIN_EVENTS}회 이상 (간격 분석 대상)': len(gap_df)}
fn = pd.Series(funnel)
fig, ax = plt.subplots(figsize=(8, 3.4))
bars = ax.barh(fn.index[::-1], fn.values[::-1], color=BLUE)
for b_, v in zip(bars, fn.values[::-1]):
    ax.text(v, b_.get_y() + b_.get_height() / 2, f' {v:,}', va='center')
ax.set_title('분석 표본이 되기까지'); ax.set_xlabel('게임 수'); ax.set_xlim(0, fn.max() * 1.15)
savefig(fig, 'a1_1_funnel.png'); plt.show()
R['n_games'], R['n_h1_games'] = int(len(series_all)), int(len(gap_df))

### 누가 분석에서 빠졌나
깔때기 차트에서 가장 많이 줄어든 곳은 마지막 단계(**할인 4회 이상**)입니다. 빠진 게임이 어떤 게임인지, 남은 게임과 무엇이 다른지 비교합니다. 빠진 게임이 한쪽으로 쏠려 있으면(예: 모두 신작) 결과를 해석할 때 조심해야 하기 때문입니다.

In [ ]:
who = games.set_index('game_id').join(n_events.rename('n_events'))
who['대상'] = np.where(who['n_events'] >= MIN_EVENTS, f'포함 (할인 {MIN_EVENTS}회 이상)', f'제외 (할인 {MIN_EVENTS}회 미만)')
who['무료 게임'] = who.index.map({gid: bool(d['price_krw'].max() == 0) for gid, d in series_all.items()})   # 가격이 계속 0원
who['출시연도'] = pd.to_datetime(who['release_date']).dt.year
who['2023년 이후 출시'] = who['출시연도'] >= 2023
who['2018년 이전 출시'] = who['출시연도'] <= 2018

compare = who.groupby('대상').agg(게임수=('n_events', 'size'), 무료게임비율=('무료 게임', 'mean'), 출시연도중앙값=('출시연도', 'median'),
                                  **{'2023년 이후 출시 비율': ('2023년 이후 출시', 'mean'), '2018년 이전 출시 비율': ('2018년 이전 출시', 'mean')},
                                  한국어리뷰수중앙값=('review_count_ko', 'median'))
display(compare.round(2))

out = who[who['n_events'] < MIN_EVENTS]
print('제외된 게임의 최근 2년 할인 횟수:', out['n_events'].value_counts().sort_index().to_dict())
R['excluded'] = dict(n=int(len(out)), free_share=float(out['무료 게임'].mean()), n_events=out['n_events'].value_counts().sort_index().to_dict())

### 표 해설
할인이 4회 이상이라 **분석에 들어간 게임**과, 4회 미만이라 **빠진 게임**의 특징을 나란히 놓은 표입니다.

### 이 표로 알 수 있는 것
- **무료 게임**이 빠진 쪽에 몰려 있습니다(분석에 들어간 쪽은 0%). 무료 게임은 할인 예측이 의미가 없으니 빠지는 것이 자연스럽습니다.
- 빠진 게임의 나머지는 대부분 최근 2년에 **할인이 0~1회뿐**인 게임입니다. 간격을 구할 수 없습니다.
- **출시연도, 2023년 이후 출시 비율, 한국어 리뷰 수가 두 집단에서 비슷하면** "신작이라서" 또는 "인기가 없어서" 빠진 것이 아닙니다. 즉 빠진 게임은 나이나 인기가 아니라 **할인을 자주 하느냐**로만 갈립니다.
- 따라서 이 노트북의 결론은 "**최근 2년에 4번 이상 할인한 게임**"에 대한 것입니다. 할인을 거의 안 하는 게임에는 적용되지 않습니다. 서비스 규칙(`timing.py`)도 할인이 2회 미만이면 "판단 불가"로 예측하지 않으므로, 빠진 게임은 서비스에서도 예측하지 않는 게임입니다.

### 의미 판단 (2026-11-26 데이터 기준)
- **깔때기**: 1,000개 → 할인 1회 이상 909개 → 4회 이상 683개. 줄어드는 곳은 마지막 두 단계뿐이고, 특히 "4회 이상" 조건에서 226개가 더 빠집니다.
- **누가 빠졌나**: 빠진 317개는 무료 게임(25%)과 최근 2년에 할인이 0~3회뿐인 게임입니다. 출시연도(중앙값 2019)와 인기(한국어 리뷰 수)는 남은 게임과 비슷합니다.
- 즉 **나이나 인기 때문이 아니라 "할인을 자주 하느냐"로만 갈립니다.** 따라서 이 노트북의 결론은 "최근 2년에 4번 이상 할인한 게임(전체의 약 68%)"에 대한 것이고, 할인을 거의 안 하는 게임에는 적용되지 않습니다. 서비스 규칙도 할인 2회 미만은 예측하지 않으므로, 빠진 게임은 서비스에서도 예측 대상이 아닙니다.

## 5. 탐색적 분석 (EDA)

In [ ]:
# 게임별 할인 횟수 (최근 2년)와 할인 시작일 사이의 간격
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(n_events, bins=range(0, int(n_events.max()) + 2), color=BLUE)
ax[0].axvline(MIN_EVENTS - .5, color=ORANGE, ls='--', label=f'간격 분석 기준 ({MIN_EVENTS}회 이상)')
ax[0].set_title('게임별 할인 횟수 (최근 2년)'); ax[0].set_xlabel('할인 횟수'); ax[0].set_ylabel('게임 수'); ax[0].legend()
ax[1].hist(pooled, bins=np.arange(0, 300, 7), color=BLUE)
ax[1].axvline(np.median(pooled), color=ORANGE, ls='--', lw=2, label=f'중앙값 {np.median(pooled):.0f}일')
ax[1].set_title('★ 할인과 할인 사이는 보통 몇 일일까?'); ax[1].set_xlabel('할인 시작일 간격(일)'); ax[1].set_ylabel('횟수'); ax[1].legend()
savefig(fig, 'a1_2_eda.png'); plt.show()
R['gap_median_days'] = float(np.median(pooled))

### 차트 해설
왼쪽은 게임마다 최근 2년에 할인이 몇 번 있었는지, 오른쪽은 모든 게임의 할인 시작일 사이 간격(일)을 합쳐 본 분포입니다.

### 이 차트로 알 수 있는 것
- 왼쪽: 할인이 아주 많은 게임과 거의 없는 게임이 섞여 있습니다. 주황 점선 왼쪽(4회 미만) 게임은 간격을 믿을 만하게 구할 수 없어 규칙성 분석에서 뺍니다.
- 오른쪽: 간격이 한두 구간에 뾰족하게 몰리면 주기성이 있다는 신호입니다. 오른쪽으로 길게 늘어진 꼬리는 가끔 오래 쉬는 경우이고, 이것이 **평균을 끌어올려** 예측을 흔드는 원인이 됩니다(6-3장).
- **키 판매점이 섞인 최저가**라서 간격이 짧게(중앙값 약 1개월) 나옵니다. Steam만 보면 어떤지는 6-2(시즌성)와 7-1(강건성)에서 확인합니다.

### 의미 판단 (2026-11-26 데이터 기준)
- **게임은 두 부류로 갈립니다.** 할인을 거의 안 하는 게임(0~3회, 약 317개)과 자주 하는 게임(중앙값 10회)입니다. "4회 이상"이라는 경계는 자의적이지 않고 데이터에 맞습니다.
- **할인 간격의 중앙값은 35일(약 한 달)이고, 15~40일 구간에 뾰족하게 몰립니다.** 주기성이 있다는 첫 신호이고, 진짜 규칙적인지는 6장에서 통계로 확인합니다.
- **오른쪽의 긴 꼬리**(가끔 100일 넘게 쉬는 경우)가 평균 간격을 끌어올립니다. 서비스가 "평균 간격"으로 예측하면 흔들리는 이유가 여기서 이미 보이고, 6-3장에서 결과로 확인됩니다.
- 간격이 한 달 정도로 짧은 것은 키 판매점까지 포함한 "모든 스토어 최저가" 기준이라서 어느 스토어든 할인하면 1회로 세기 때문입니다. 이 분석의 주요 한계이고 6-2와 7-1에서 Steam만 따로 확인합니다.

## 6. 가설 검정

공통 도구를 먼저 정의합니다. (판정 함수, 게임 단위 부트스트랩 신뢰구간)

In [ ]:
def verdict(p, effect_ok, direction_ok=True):
    if direction_ok and p < ALPHA and effect_ok:
        return '✅ 가설과 일치 — 통계적으로 유의하고 효과도 실무 기준을 넘음'
    if direction_ok and p < ALPHA:
        return '⚠️ 부분 일치 — 통계적으로는 유의하나 효과가 실무 기준에 못 미침'
    return '❌ 불일치 — 귀무가설을 기각하지 못함(차이가 있다고 말할 근거 부족) 또는 방향이 반대'

def mark(ok):
    return {True: '✅ 일치', False: '❌ 불일치', None: '검정 안 함'}[ok]

def mark_r(d, p_key='p'):
    # 효과가 실무 기준을 넘으면 일치, 통계적으로는 유의한데 효과가 기준에 못 미치면 부분 일치, 그 밖에는 불일치
    if d['ok']:
        return mark(True)
    return '⚠️ 부분 일치' if d.get(p_key, 1.0) < ALPHA else mark(False)

def cluster_diff_ci(bt, a, b, B=N_BOOT, seed=SEED):
    """같은 게임의 예측은 비슷하게 움직이므로, 게임을 단위로 다시 뽑아 평균 오차 차이(a-b)의 신뢰구간을 구합니다."""
    g = bt.assign(d=bt[a] - bt[b]).groupby('game_id')['d'].agg(['sum', 'count'])
    s, c = g['sum'].to_numpy(), g['count'].to_numpy()
    rng = np.random.default_rng(seed)
    vals = [s[i].sum() / c[i].sum() for i in (rng.integers(0, len(g), len(g)) for _ in range(B))]
    return np.percentile(vals, [2.5, 97.5])

### 6-1) H1. 할인은 규칙적인가? ★

* **변동계수(CV)** = 간격의 흔들림 ÷ 평균 간격. 작을수록 규칙적입니다.
* 할인이 **무작위**라면 CV는 1 안팎입니다. 다만 간격 수가 적으면 우연히 작게 나오므로, 게임마다 **같은 횟수의 무작위 할인**을 시뮬레이션해 기준값(`cv_null`)으로 삼았습니다.
* **① 정규성 확인 → 도구 선택**: (실제 CV - 무작위 기준 CV)가 정규분포가 아니면 t-검정 대신 순위 기반 **Wilcoxon 부호순위검정**(같은 게임의 짝지은 자료, 단측)을 씁니다.
* **② 신뢰구간**: 게임을 단위로 다시 뽑아 차이의 중앙값 신뢰구간을 구합니다.

In [ ]:
diff = gap_df['cv'] - gap_df['cv_null']                        # 음수 = 무작위보다 규칙적
sw = stats.shapiro(diff.to_numpy()[:5000])
print(f'Shapiro-Wilk 정규성 검정(CV 차이): W={sw.statistic:.3f}, p={sw.pvalue:.4g} →',
      '정규분포가 아님 → 비모수 검정 사용' if sw.pvalue < ALPHA else '정규성을 부정할 근거 없음(그래도 비모수 검정으로 보수적으로 확인)')

w1 = stats.wilcoxon(diff, alternative='less')
share_regular = float((diff < 0).mean())
med_diff = float(diff.median())
rng = np.random.default_rng(SEED)
vals = [np.median(diff.to_numpy()[rng.integers(0, len(diff), len(diff))]) for _ in range(N_BOOT)]
lo1, hi1 = np.percentile(vals, [2.5, 97.5])
h1_effect_ok = bool(med_diff <= -PRACTICAL_CV and share_regular >= 0.5 and hi1 < 0)

h1 = pd.DataFrame({'값': [len(gap_df), f"{gap_df['cv'].median():.2f}", f"{gap_df['cv_null'].median():.2f}", f'{med_diff:+.2f}',
                          f'{lo1:+.2f} ~ {hi1:+.2f}', f'{share_regular:.1%}', f'{w1.pvalue:.2e}']},
                  index=['게임 수', '실제 CV 중앙값', '무작위 기준 CV 중앙값', '차이(실제-무작위) 중앙값', '차이의 95% 신뢰구간(게임 단위 부트스트랩)',
                         '무작위보다 규칙적인 게임 비율', 'Wilcoxon p (단측)'])
display(h1)
R['h1'] = dict(n_games=len(gap_df), cv_median=float(gap_df['cv'].median()), cv_null_median=float(gap_df['cv_null'].median()),
               median_diff=med_diff, ci=[float(lo1), float(hi1)], share_regular=share_regular, p=float(w1.pvalue), ok=h1_effect_ok)
print('\nH1 판정:', verdict(w1.pvalue, h1_effect_ok, direction_ok=med_diff < 0))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(gap_df['cv'], bins=30, color=BLUE)
ax.axvline(gap_df['cv_null'].median(), color=ORANGE, ls='--', lw=2, label=f"무작위라면 약 {gap_df['cv_null'].median():.2f}")
ax.axvline(gap_df['cv'].median(), color='k', lw=2, label=f"실제 중앙값 {gap_df['cv'].median():.2f}")
ax.set_title(f'★ 게임의 {share_regular:.1%}가 무작위보다 규칙적'); ax.set_xlabel('할인 간격의 흔들림 (CV, 작을수록 규칙적)'); ax.set_ylabel('게임 수'); ax.legend()
savefig(fig, 'a1_3_h1.png'); plt.show()

### 차트 해설
게임별 할인 간격의 흔들림(CV) 분포입니다. 주황 점선은 "할인이 무작위였다면 나왔을 값", 검은 선은 실제 중앙값입니다.

### 이 차트로 알 수 있는 것
- 막대 대부분이 주황 점선의 **왼쪽**에 있으면 무작위보다 규칙적이라는 뜻입니다. 검은 선이 주황 점선보다 0.2 이상 왼쪽에 있으면 실무 기준을 넘습니다.
- 오른쪽 끝에 남은 게임은 할인 간격이 들쭉날쭉한 게임입니다. "규칙적"이 모든 게임에 해당하지는 않습니다.

### 의미 판단 (2026-11-26 데이터 기준)
- **할인에는 확실한 리듬이 있습니다.** 실제 CV 중앙값 0.54는 무작위일 때의 0.94보다 훨씬 작고, 683개 게임 중 85.5%가 무작위보다 규칙적입니다.
- 차이의 신뢰구간(−0.42 ~ −0.38)이 좁고 0에서 멀어서 우연으로는 설명이 안 됩니다. 실무 기준(0.2 이상 작음)도 크게 넘어 **✅ 가설과 일치**합니다.
- 의미: "다음 할인을 예측해 볼 만한 근거가 있다"는 뜻입니다. 다만 막대 오른쪽 끝의 게임처럼 불규칙한 게임도 있어서 모든 게임에 해당하는 것은 아닙니다.

### 6-2) H2. 할인은 특정 달에 몰리는가?

H2는 네 단계로 확인합니다. **(a) 모든 스토어의 최저가** → **(b) Steam만** → **(c) 몰린 날짜가 정확히 언제인가** → **(d) 왜 달라지나(스토어별)**. 결론은 6-2e에 있습니다.

#### 6-2a) 모든 스토어의 최저가로 본 시즌성
달별 **할인 시작 횟수**가 고른지 **카이제곱 적합도 검정**을 합니다. 달마다 관찰한 날수가 달라서(예: 24개월이면 어떤 달은 2번, 어떤 달은 1번 관찰), 기대 횟수는 **관찰한 날수 비율**로 잡았습니다.
효과 크기는 **Cramér's V**로 보고, 게임을 단위로 다시 뽑아 신뢰구간을 구합니다.

In [ ]:
def month_matrices(series, ev):
    """게임 × 12개월 행렬 두 개: 관찰한 날수(exposure), 할인 시작 횟수(observed)"""
    gids = list(series)
    expo, obs = np.zeros((len(gids), 12)), np.zeros((len(gids), 12))
    for i, gid in enumerate(gids):
        span = pd.date_range(max(series[gid]['day'].min(), SINCE), END)
        expo[i] = np.bincount(span.month - 1, minlength=12)
        obs[i] = np.bincount(ev[gid]['start'].dt.month - 1, minlength=12)
    return obs, expo

def cramers_v(obs, expo):
    expected = expo / expo.sum() * obs.sum()
    chi2, p = stats.chisquare(obs, expected)
    return np.sqrt(chi2 / (obs.sum() * 11)), chi2, p, expected

obs_g, expo_g = month_matrices(series_all, events)
observed, exposure = obs_g.sum(axis=0), expo_g.sum(axis=0)
v, chi2, p_month, expected = cramers_v(observed, exposure)
rng = np.random.default_rng(SEED)
vs = [cramers_v(obs_g[i].sum(axis=0), expo_g[i].sum(axis=0))[0] for i in (rng.integers(0, len(obs_g), len(obs_g)) for _ in range(N_BOOT))]
lo2, hi2 = np.percentile(vs, [2.5, 97.5])
ratio = pd.Series(observed / expected, index=range(1, 13))
busy = [int(m) for m, r in ratio.items() if r > 1.15]
h2_effect_ok = bool(v >= PRACTICAL_V)

print(f'할인 시작 {int(observed.sum()):,}건 · 카이제곱 {chi2:.1f}(자유도 11), p = {p_month:.3g}')
print(f"Cramér's V = {v:.3f} (95% 신뢰구간 {lo2:.3f} ~ {hi2:.3f}, 게임 단위 부트스트랩)  ←  실무 기준 {PRACTICAL_V}")
print('평균보다 15% 이상 많은 달:', busy)
R['h2'] = dict(chi2=float(chi2), p=float(p_month), v=float(v), ci=[float(lo2), float(hi2)], busy_months=busy,
               ratio={int(k): float(x) for k, x in ratio.items()}, ok=bool(p_month < ALPHA and h2_effect_ok))
print('\nH2-a (모든 스토어) 판정:', verdict(p_month, h2_effect_ok))

fig, ax = plt.subplots(figsize=(9, 3.8))
ratio.plot.bar(color=np.where(ratio > 1.15, RED, '#9ecae9'), width=.8, ax=ax)
ax.axhline(1, color='k', lw=1)
ax.set_title('월별 할인 시작 횟수 (1 = 평균)'); ax.set_xlabel('월'); ax.set_ylabel('평균 대비 배율'); plt.xticks(rotation=0)
savefig(fig, 'a1_4_h2.png'); plt.show()

### 차트 해설
달별로 할인이 시작된 횟수를 "그 달을 관찰한 날수로 기대한 횟수"로 나눈 배율입니다. 1이 평균입니다.

### 이 차트로 알 수 있는 것
- 막대가 1(검은 선)보다 높으면 그 달에 할인이 평소보다 자주 시작됩니다. 빨간 달(평균보다 15% 이상)이 스토어의 정기 세일 시기와 겹치는지 봅니다.
- p값이 극히 작게 나와도 **Cramér's V가 0.09보다 작으면 효과가 기준에 못 미친다**고 봅니다(Cohen의 '중간 효과'에 해당하는 값). 표본이 크고 게임들이 같은 세일에 묶여 있어 p값은 쉽게 작아지기 때문입니다.

### 의미 판단 (2026-11-26 데이터 기준)
- **3·6·12월에 할인이 몰리는 것은 사실입니다.** 스토어의 봄·여름·겨울 세일 시기와 겹치고, 12월이 평균의 약 1.6배로 가장 많습니다.
- 하지만 **쏠림 정도는 기준에 못 미칩니다.** Cramér's V가 0.082로 실무 기준(0.09)보다 작아서 **⚠️ 부분 일치**입니다. p값이 1e-160으로 극히 작은 것은 표본이 1만 건이 넘고 게임들이 같은 세일에 묶여 있기 때문이므로, p값이 아니라 V로 판단해야 합니다. (12월이 평균의 1.6배이니 쏠림이 없는 것은 아니고, 뚜렷하다고 하기에는 약하다는 뜻입니다.)
- **이 약함이 진짜 약해서인지, 상시 할인 판매점이 섞여서 희석된 것인지는 아직 모릅니다.** 6-2b부터 확인합니다.

#### 6-2b) Steam만으로 다시 보기 (H2-b)
모든 스토어의 최저가에는 상시 할인에 가까운 판매점도 섞여 있습니다. 같은 월별 계산을 **Steam(공식 스토어) 가격만으로** 다시 해서 나란히 비교합니다.

In [ ]:
series_steam = make_series(prices[prices['store'] == 'steam'])        # Steam(공식 스토어) 가격만으로 만든 일별 최저가
ev_steam = make_events(series_steam)
obs_gs, expo_gs = month_matrices(series_steam, ev_steam)                 # 게임 x 12개월 (Steam만)
v_s, chi2_s, p_s, expected_s = cramers_v(obs_gs.sum(axis=0), expo_gs.sum(axis=0))
ratio_s = pd.Series(obs_gs.sum(axis=0) / expected_s, index=range(1, 13))
busy_s = [int(m) for m, r in ratio_s.items() if r > 1.15]
rng = np.random.default_rng(SEED)
vs_s = [cramers_v(obs_gs[i].sum(axis=0), expo_gs[i].sum(axis=0))[0] for i in (rng.integers(0, len(obs_gs), len(obs_gs)) for _ in range(N_BOOT))]
lo_s, hi_s = np.percentile(vs_s, [2.5, 97.5])
v_all = R['h2']['v']

print(f"모든 스토어: V = {v_all:.3f}, 몰리는 달 {R['h2']['busy_months']}")
print(f"Steam만    : V = {v_s:.3f} (95% 신뢰구간 {lo_s:.3f} ~ {hi_s:.3f}), 몰리는 달 {busy_s}, 카이제곱 p = {p_s:.3g}  ←  실무 기준 {PRACTICAL_V}")
h2s_effect_ok = bool(v_s >= PRACTICAL_V)
R['h2_steam'] = dict(v=float(v_s), ci=[float(lo_s), float(hi_s)], p=float(p_s), busy_months=busy_s, ratio={int(k): float(x) for k, x in ratio_s.items()},
                     ok=bool(p_s < ALPHA and h2s_effect_ok))
print('\nH2-b (Steam만) 판정:', verdict(p_s, h2s_effect_ok))

fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for a_, rr, title, vv in ((ax[0], ratio, '모든 스토어 최저가', v_all), (ax[1], ratio_s, 'Steam만 (공식 스토어)', v_s)):
    rr.plot.bar(color=np.where(rr > 1.15, RED, '#9ecae9'), width=.8, ax=a_)
    a_.axhline(1, color='k', lw=1)
    a_.set_title(f'{title}  (Cramér V = {vv:.2f})'); a_.set_xlabel('월'); a_.tick_params(axis='x', labelrotation=0)
ax[0].set_ylabel('평균 대비 배율 (1 = 평균)')
savefig(fig, 'a1_6b_h2_steam.png'); plt.show()

### 차트 해설
왼쪽은 6-2와 같은 차트(모든 스토어 최저가), 오른쪽은 같은 계산을 Steam 가격만으로 한 것입니다. 막대는 "그 달에 할인이 시작된 횟수 ÷ 그 달을 관찰한 날수로 기대한 횟수"이고, 1이 평균, 빨강은 평균보다 15% 이상 많은 달입니다.

### 이 차트로 알 수 있는 것
- 오른쪽(Steam)에서 막대 높이의 들쭉날쭉함이 왼쪽보다 크면 **공식 스토어의 시즌 쏠림이 더 뚜렷하다**는 뜻입니다. 제목의 Cramér V가 클수록 쏠림이 큽니다.
- 두 차트에서 **빨간 달이 같은지 다른지** 비교합니다. 같으면 같은 시기에 몰리는 것이고, 다르면 스토어마다 할인 일정이 다르다는 뜻입니다.

### 의미 판단 (2026-11-26 데이터 기준)
- **두 차트에서 같은 달(3·6·12월)이 빨강입니다.** 어느 스토어를 보든 봄·여름·겨울 세일 시기에 할인이 몰린다는 큰 그림은 같습니다.
- 하지만 **Steam은 쏠림이 훨씬 뚜렷합니다.** 3월 1.5배, 6월 1.7배, 12월 1.7배로, 모든 스토어 기준(3월 1.2배, 6월 1.4배, 12월 1.6배)보다 높고, 비수기는 더 낮습니다(7월은 평균의 절반인 0.5배). Cramér V가 **0.08 → 0.13**(신뢰구간 0.125~0.132)이 되어 실무 기준 0.09를 넘고 **✅ 가설과 일치(H2-b)** 합니다. 두 V의 신뢰구간이 겹치지 않아 이 차이는 우연이 아닙니다.
- Steam에서는 **9월(1.2배)과 10월(1.15배)도** 평균보다 15% 이상 많습니다(모든 스토어에서는 약 1.0배). 이유는 바로 아래 날짜별 표에서 확인합니다(가을 세일이 해마다 다른 달에 시작했습니다).
- 주의: 이력이 약 24개월이라 각 달이 1~2번만 관찰됩니다. 연도에 따라 달라질 수 있습니다.

#### 6-2c) 어느 날짜에 몰렸나 — Steam 세일 일정과 대조
월 단위 차트에서 몰린 달이 **정확히 어느 날짜**인지 보고, 그날이 Steam의 어떤 세일인지 맞춰 봅니다. 하루에 **게임의 30% 이상**이 같은 날 할인을 시작한 날만 골랐습니다. 세일 이름과 기간은 Valve가 공지한 일정을 보도한 기사 기준입니다(출처는 아래 해설).

In [ ]:
SALES = [  # (Steam 세일 이름, 시작일, 종료일) - Valve 공지를 보도한 기사 기준
    ('겨울 세일 (Winter Sale)', '2024-12-19', '2025-01-02'),
    ('봄 세일 (Spring Sale)', '2025-03-13', '2025-03-20'),
    ('여름 세일 (Summer Sale)', '2025-06-26', '2025-07-10'),
    ('가을 세일 (Autumn Sale)', '2025-09-29', '2025-10-06'),
    ('겨울 세일 (Winter Sale)', '2025-12-18', '2026-01-05'),
    ('봄 세일 (Spring Sale)', '2026-03-19', '2026-03-26'),
    ('여름 세일 (Summer Sale)', '2026-06-25', '2026-07-09'),
    ('가을 세일 (Autumn Sale)', '2026-10-01', '2026-10-08')]
sale_by_start = {pd.Timestamp(st): (name, f'{st} ~ {en}') for name, st, en in SALES}

starts_all = pd.concat([e['start'] for e in ev_steam.values()])           # Steam 할인 시작일 전체
daily_n = starts_all.value_counts().sort_index()                           # 날짜별로 할인을 시작한 게임 수
BIG_DAY = 0.3 * len(series_steam)                                          # 게임의 30% 이상이 같은 날 시작한 날
big = daily_n[daily_n >= BIG_DAY]

rows = []
for d_, n_ in big.items():
    name, period = sale_by_start.get(d_, ('(일치하는 세일 없음)', ''))
    rows.append({'날짜': f'{d_:%Y-%m-%d}', '그날 할인을 시작한 게임': int(n_), '게임 대비(%)': n_ / len(series_steam) * 100,
                 'Steam 세일': name, '기간': period})
sale_tab = pd.DataFrame(rows)
display(sale_tab.round(1))

share_big = big.sum() / len(starts_all) * 100
matched = int(sum(pd.Timestamp(r['날짜']) in sale_by_start for r in rows))
print(f'하루에 게임의 30% 이상이 할인을 시작한 날: {len(big)}일 → Steam 할인 시작 전체({len(starts_all):,}건)의 {share_big:.0f}%가 이 {len(big)}일에 시작')
print(f'그중 Steam 정기 세일 시작일과 일치하는 날: {matched}일')
R['steam_big_days'] = dict(n_days=int(len(big)), share_of_starts=float(share_big), matched=matched, days=rows)

fig, ax = plt.subplots(figsize=(13, 4.2))
ax.bar(daily_n.index, daily_n.values, width=1.0, color=GRAY)
ax.bar(big.index, big.values, width=4, color=RED)
for d_, n_ in big.items():
    name = sale_by_start.get(d_, ('?',))[0].split(' (')[0]
    ax.annotate(f'{name}\n{d_:%y/%m/%d}', (d_, n_), ha='center', va='bottom', fontsize=8)
ax.set(title='Steam: 날짜별로 할인을 시작한 게임 수 (빨강 = 게임의 30% 이상이 같은 날 시작)', ylabel='게임 수', ylim=(0, daily_n.max() * 1.25))
savefig(fig, 'a1_6c_steam_sale_days.png'); plt.show()

### 차트 해설
회색 막대는 날짜마다 "그날 할인을 시작한 게임이 몇 개인가"이고, 빨강은 그중 게임의 30% 이상이 같은 날 시작한 날입니다. 위 표는 그 날짜들과 Steam 세일 이름을 맞춘 것입니다.

### 이 차트로 알 수 있는 것
- 빨간 막대는 평소(회색, 하루 수십~백여 개)와 비교가 안 될 만큼 **하루에 수백 개가 한꺼번에** 시작한 날입니다.
- 빨간 날이 Steam 정기 세일 시작일과 일치하면, 월별 쏠림은 **정기 세일이 만든 것**입니다.
- 출처: [VideoGamer(2024 겨울 세일 12/19~1/2)](https://www.videogamer.com/news/steam-winter-sale-2024-start-time-countdown), [PC Gamer(가을 세일을 11월 말에서 앞당김)](https://pcgamer.com/games/after-years-of-running-awkwardly-close-to-the-winter-sale-the-steam-autumn-sale-will-start-2-months-early-this-year), [Game8(2025 일정)](https://game8.co/articles/latest/steam-sales-schedule-for-2025-officially-revealed), [ComicBook(2025 겨울 세일)](https://comicbook.com/gaming/news/steam-sale-schedule-next-steam-sale-dates), [Notebookcheck(2026 일정)](https://www.notebookcheck.net/Gamers-can-mark-their-calendars-All-important-Steam-events-in-2026.1207386.0.html)

### 의미 판단 (2026-11-26 데이터 기준)
- **Steam의 시즌 쏠림은 4대 정기 세일이 만듭니다.** 하루에 게임의 약 6~7할(586~717개)이 한꺼번에 할인을 시작한 날이 8일 있고, 모두 Steam의 봄·여름·가을·겨울 세일 시작일과 일치합니다. 이 8일에 Steam 할인 시작 전체의 약 38%가 몰려 있습니다.
- **9월과 10월이 둘 다 높고 11월이 낮은 이유**: Valve가 **가을 세일을 원래의 11월 말에서 앞당겼습니다.** 2025년에는 9월 29일, 2026년에는 10월 1일에 시작해서, 가을 세일이 해마다 9월과 10월로 갈라졌습니다. 11월에는 더 이상 큰 세일이 없습니다.
- **3·6·12월이 가장 높은 이유**: 데이터 기간(2024-12 ~ 2026-11)에 봄·여름·겨울 세일이 각각 두 번씩 들어 있어서입니다. 가을 세일은 9월에 1번, 10월에 1번이라 두 달로 나뉘어 각각 덜 높게 보입니다.
- **서비스에 주는 뜻**: 이 4대 세일의 시작일은 Valve가 **미리 공지**하므로, Steam 가격 기준의 "다음 할인"은 과거 패턴으로 예측하는 것보다 **공지된 세일 일정으로 안내**하는 편이 정확합니다. 다음은 2026년 겨울 세일(12월 17일 ~ 2027년 1월 4일)입니다.
- **주의**: 가을 세일이 두 달 앞당겨진 것처럼 일정은 해마다 바뀔 수 있습니다. 과거 몇 년의 "월별 패턴"으로 미래의 달을 단정하면 안 되고, **그해의 공식 일정**을 봐야 합니다. 이력이 24개월뿐이라는 한계도 그대로입니다.

#### 6-2d) 왜 달라지나 — 스토어를 섞으면 쏠림이 희석되는가
모든 스토어 기준에서 쏠림이 약했던 이유를 두 가지로 확인합니다.
1. **할인 시작 건수**: 모든 스토어의 최저가는 어느 스토어에서든 이미 할인 중이면 "할인 중" 상태가 이어집니다. 그러면 다른 스토어의 정기 세일 시작이 **새 할인으로 안 잡혀서** 시작 건수가 줄고 쏠림이 흐려집니다. 스토어를 섞었는데 시작 건수가 줄었다면 이 현상의 증거입니다.
2. **스토어별 쏠림(V)과 상시 할인 정도**: 스토어마다 따로 계산합니다. "상시 할인 정도"는 최근 365일 중 할인 중이었던 날의 비율(게임별 중앙값)입니다. 1에 가까우면 거의 1년 내내 할인하는 판매점입니다.

In [ ]:
# (1) 스토어를 섞으면 "할인 시작" 건수가 오히려 줄어든다
n_steam_starts, n_all_starts = int(len(starts_all)), int(observed.sum())
print(f'할인 시작 건수: Steam만 {n_steam_starts:,}건  vs  모든 스토어 최저가 {n_all_starts:,}건  →  스토어를 섞었는데 오히려 {1 - n_all_starts / n_steam_starts:.0%} 적음')
R['h2_dilution'] = dict(steam_starts=n_steam_starts, all_starts=n_all_starts)

# (2) 스토어별 시즌 쏠림(Cramér's V)과 상시 할인 정도
YEAR_DAYS, MIN_STORE_GAMES, MIN_STORE_STARTS = 365, 250, 2000

def store_row(name, ser):
    ev_st = make_events(ser)
    ob_st, ex_st = month_matrices(ser, ev_st)
    v_st, _, p_st, exp_st = cramers_v(ob_st.sum(axis=0), ex_st.sum(axis=0))
    shares = []                                                      # 최근 365일 중 할인 중인 날의 비율 (게임별)
    for d in ser.values():
        cal = d.set_index('day')['discount_pct'].reindex(pd.date_range(d['day'].min(), END)).ffill()
        if cal.index.min() <= END - pd.Timedelta(days=YEAR_DAYS - 1):
            shares.append((cal.loc[END - pd.Timedelta(days=YEAR_DAYS - 1):END] >= SALE_MIN_PCT).mean())
    return {'스토어': name, '게임 수': len(ser), '할인 시작 건수': int(ob_st.sum()), "Cramér's V": v_st,
            '상시 할인 정도': float(np.median(shares)) if shares else np.nan,
            '몰리는 달': [int(m) + 1 for m, r in enumerate(ob_st.sum(axis=0) / exp_st) if r > 1.15]}

store_games = prices.groupby('store')['game_id'].nunique()
rows = [store_row(st, make_series(prices[prices['store'] == st])) for st in store_games[store_games >= MIN_STORE_GAMES].index]
store_tab = pd.DataFrame(rows)
store_tab = store_tab[store_tab['할인 시작 건수'] >= MIN_STORE_STARTS]       # 할인 시작이 너무 적은 스토어는 V가 불안정해서 뺀다
store_tab = pd.concat([store_tab, pd.DataFrame([store_row('[모든 스토어 최저가]', series_all)])]).sort_values("Cramér's V", ascending=False)
display(store_tab.round(3).reset_index(drop=True))
R['h2_stores'] = store_tab.round(4).to_dict('records')
print(f'※ 게임 {MIN_STORE_GAMES}개 이상, 할인 시작 {MIN_STORE_STARTS:,}건 이상인 스토어만 표시')

is_total = store_tab['스토어'].str.startswith('[')
rho_store, p_store = stats.spearmanr(store_tab.loc[~is_total, '상시 할인 정도'], store_tab.loc[~is_total, "Cramér's V"])
print(f"스토어 간 상관(상시 할인 정도 vs 시즌 쏠림 V): Spearman ρ = {rho_store:+.2f}, p = {p_store:.2f} (스토어 {int((~is_total).sum())}개)")
R['h2_store_corr'] = dict(rho=float(rho_store), p=float(p_store), n=int((~is_total).sum()))

fig, ax = plt.subplots(figsize=(9, 4.4))
st_plot = store_tab.iloc[::-1]
colors = [RED if n == 'steam' else (GRAY if n.startswith('[') else BLUE) for n in st_plot['스토어']]
ax.barh(st_plot['스토어'], st_plot["Cramér's V"], color=colors)
ax.axvline(PRACTICAL_V, color=ORANGE, ls='--', label=f'실무 기준 {PRACTICAL_V}')
ax.set(title="스토어별 시즌 쏠림 (Cramér's V) — 빨강 Steam, 회색 모든 스토어 최저가", xlabel="Cramér's V"); ax.legend()
savefig(fig, 'a1_6d_store_season.png'); plt.show()

### 표·차트 해설
첫 줄은 Steam만 볼 때와 모든 스토어의 최저가로 볼 때의 **할인 시작 건수**입니다. 아래 표는 스토어마다 따로 계산한 시즌 쏠림(V), 상시 할인 정도, 몰리는 달이고, 막대는 V를 비교한 것입니다. 주황 점선이 실무 기준(0.09)입니다.

### 이 표로 알 수 있는 것
- 스토어를 섞었는데 할인 시작 건수가 **줄었다면**, 어느 스토어든 이미 할인 중이라 다른 스토어의 새 할인이 새 시작으로 안 잡힌다는 뜻입니다(희석).
- **상시 할인 정도가 높은 스토어**(1에 가까움)에서 V가 낮으면, "상시 할인 판매점이 쏠림을 흐린다"는 가설과 맞습니다.
- 반대로 **상시 할인 정도가 낮은데도 V가 낮은 스토어**나 **V가 높은 다른 스토어**가 있으면 "Steam만 특별하다"거나 "공식 스토어는 모두 강하다"고 일반화할 수 없다는 뜻입니다.

### 의미 판단 (2026-11-26 데이터 기준)
- **스토어를 섞으면 할인 시작이 줄어듭니다.** Steam만 14,126건인데 모든 스토어의 최저가로 보면 10,685건으로 **24% 적습니다.** 스토어를 더 많이 합쳤는데 시작이 줄어든 것은, 어느 스토어에서든 이미 할인 중이면 다른 스토어의 정기 세일이 **새 할인으로 안 잡히기 때문**입니다. 표의 "상시 할인 정도"도 같은 이야기입니다. 모든 스토어의 최저가는 1년의 약 79%가 할인 중인데 Steam은 28%뿐입니다.
- 그래서 **모든 스토어 기준의 쏠림이 약했던 것은 쏠림이 없어서가 아니라, 상시 할인에 가까운 판매점이 섞여 희석됐기 때문**이라는 설명이 두 가지 증거(시작 건수 감소, 거의 매일 할인 중인 최저가)로 뒷받침됩니다.
- **하지만 스토어별로 보면 "공식 스토어는 강하고 상시 할인 판매점은 약하다"로 깔끔하게 나뉘지는 않습니다.**
  - **Steam(V 0.128)뿐 아니라 Epic(0.197)도** 기준(0.09)을 넘어 뚜렷합니다.
  - 거의 1년 내내 할인 중인 판매점(상시 할인 정도 0.95 이상)인 Fanatical은 V가 0.065로 낮고, GameBillet은 0.091로 기준 근처입니다.
  - 그런데 **상시 할인 정도가 낮은 스토어도 V가 낮습니다**(Humble Store 0.27→V 0.078, GamersGate 0.32→V 0.063). 또 GOG(0.060), Ubisoft(0.071)처럼 상시 할인과 거리가 먼 스토어도 낮습니다.
  - 스토어 10개에서 "상시 할인 정도"와 V의 상관은 **ρ = -0.44(p = 0.20)** 로 방향은 맞지만 약하고 통계적으로 확정되지 않습니다.
- 의미: 쏠림이 뚜렷한 곳은 "공식 스토어 전체"가 아니라 **Steam처럼 정기 세일에 많은 게임이 한꺼번에 참여하는 스토어**입니다(Epic도 같은 패턴인지는 바로 아래에서 확인합니다).
- 주의: 표에는 게임 250개 이상, 할인 시작 2,000건 이상인 스토어만 넣었습니다(그보다 적으면 V가 불안정). 스토어 수가 10개뿐이라 상관은 참고용입니다.

#### 6-2d 참고) Epic의 정기 세일 날짜 — Steam보다 V가 높은 이유
위 표에서 Epic(V 0.197)이 Steam(0.128)보다 높습니다. 같은 방식으로 **Epic에서 몰린 날짜**를 보고 Steam과 비교합니다.

In [ ]:
EPIC_SALES = [  # (Epic 세일 이름, 시작일, 종료일) - Epic 공식 안내·보도 기준
    ('홀리데이 세일', '2024-12-12', '2025-01-09'),
    ('윈터 세일', '2025-02-10', '2025-02-24'),
    ('스프링 세일', '2025-03-26', '2025-04-09'),
    ('메가 세일', '2025-05-15', '2025-06-12'),
    ('서머 세일', '2025-07-17', '2025-07-31'),
    ('엔드 오브 서머 세일', '2025-09-04', '2025-09-18'),
    ('홀리데이 세일', '2025-12-11', '2026-01-08'),
    ('윈터 세일', '2026-02-09', '2026-02-23'),
    ('스프링 세일', '2026-03-30', '2026-04-13'),
    ('메가 세일', '2026-05-14', '2026-06-11'),
    ('서머 세일', '2026-07-16', '2026-07-30'),
    ('엔드 오브 서머 세일', '2026-09-03', '2026-09-17'),
    ('핼러윈 세일', '2026-10-19', '2026-11-02')]
epic_by_start = {pd.Timestamp(d_): (n_, f'{d_} ~ {e_}') for n_, d_, e_ in EPIC_SALES}

series_epic = make_series(prices[prices['store'] == 'epic'])
ev_epic = make_events(series_epic)
starts_epic = pd.concat([e['start'] for e in ev_epic.values()])
daily_epic = starts_epic.value_counts().sort_index()
big_epic = daily_epic[daily_epic >= 0.3 * len(series_epic)]                  # 게임의 30% 이상이 같은 날 시작한 날

display(pd.DataFrame([{'날짜': f'{d_:%Y-%m-%d}', '그날 할인을 시작한 게임': int(n_), '게임 대비(%)': n_ / len(series_epic) * 100,
                       'Epic 세일': epic_by_start.get(d_, ('(이름 확인 안 됨)', ''))[0], '기간': epic_by_start.get(d_, ('', ''))[1]}
                      for d_, n_ in big_epic.items()]).round(1))

ob_e, ex_e = month_matrices(series_epic, ev_epic)
v_e, _, p_e, exp_e = cramers_v(ob_e.sum(axis=0), ex_e.sum(axis=0))
ratio_e = pd.Series(ob_e.sum(axis=0) / exp_e, index=range(1, 13))
share_big_e = big_epic.sum() / len(starts_epic) * 100
display(pd.DataFrame({
    'Steam': [f'{v_s:.3f}', f'{len(big)}일', f'{big.min() / len(series_steam):.0%} ~ {big.max() / len(series_steam):.0%}', f'{share_big:.0f}%', f'{ratio_s.min():.2f} ~ {ratio_s.max():.2f}'],
    'Epic': [f'{v_e:.3f}', f'{len(big_epic)}일', f'{big_epic.min() / len(series_epic):.0%} ~ {big_epic.max() / len(series_epic):.0%}', f'{share_big_e:.0f}%', f'{ratio_e.min():.2f} ~ {ratio_e.max():.2f}']},
    index=["Cramér's V (월별 쏠림)", '큰 세일 날짜 수', '그날 할인을 시작하는 게임 비율', '큰 세일 날짜에 몰린 할인 시작 비중', '월별 배율 범위 (최저 ~ 최고)']))
R['epic'] = dict(v=float(v_e), n_big_days=int(len(big_epic)), share_of_starts=float(share_big_e), ratio={int(k): float(x) for k, x in ratio_e.items()})

fig, ax = plt.subplots(figsize=(13, 3.8))
ax.bar(daily_epic.index, daily_epic.values, width=1.0, color=GRAY)
ax.bar(big_epic.index, big_epic.values, width=4, color=RED)
for d_, n_ in big_epic.items():
    ax.annotate(f"{epic_by_start.get(d_, ('(이름 확인 안 됨)',))[0]}\n{d_:%y/%m/%d}", (d_, n_), ha='center', va='bottom', fontsize=7)
ax.set(title='Epic: 날짜별로 할인을 시작한 게임 수 (빨강 = 게임의 30% 이상이 같은 날 시작)', ylabel='게임 수', ylim=(0, daily_epic.max() * 1.3))
savefig(fig, 'a1_6e_epic_sale_days.png'); plt.show()

### 표·차트 해설
첫 표는 Epic에서 게임의 30% 이상이 같은 날 할인을 시작한 날과 Epic 세일 이름입니다. 둘째 표는 Steam과 Epic의 쏠림을 네 가지 방식으로 비교한 것입니다. 일정 출처: [Epic 스프링 세일 2025](https://store.epicgames.com/news/epic-games-store-spring-sale-2025-buyers-guide), [메가 세일 2025](https://store.epicgames.com/news/epic-games-store-mega-sale-2025), [서머 세일 2025](https://store.epicgames.com/news/epic-games-store-summer-sale-2025-buyers-guide), [엔드 오브 서머 세일 2025](https://store.epicgames.com/news/epic-games-store-end-of-summer-sale-2025-buyers-guide-best-deals), [홀리데이 세일](https://store.epicgames.com/news/epic-games-holiday-sale-buyers-guide), [스프링 세일 2026](https://store.epicgames.com/news/epic-games-spring-sale-2026-buyers-guide), [윈터 세일 2026](https://store.epicgames.com/news/epic-games-winter-sale-2026-buyers-guide), [홀리데이 세일 2024](https://store.epicgames.com/news/epic-games-store-holiday-sale-2024-buyers-guide), [윈터 세일 2025](https://store.epicgames.com/news/grab-some-hot-bargains-in-our-epic-games-store-2025-winter-sale-buyer-s-guide), [서머 세일 2026](https://store.epicgames.com/news/epic-games-summer-sale-2026-buyers-guide?lang=en-US), [엔드 오브 서머 세일 2026](https://store.epicgames.com/news/epic-games-end-of-summer-sale-2026-buyers-guide), [핼러윈 세일 2026](https://store.epicgames.com/news/epic-games-store-halloween-sale-buyers-guide-2026)

### 이 표로 알 수 있는 것
- Steam처럼 Epic도 **정기 세일 시작일에 게임이 한꺼번에** 할인을 시작합니다.
- 두 스토어의 쏠림을 **네 가지 방식**으로 나란히 봅니다. V가 높다고 세일 한 번의 규모가 큰 것은 아닙니다.

### 의미 판단 (2026-11-26 데이터 기준)
- **Epic도 정기 세일 시작일에 몰립니다.** 13일에 Epic 게임의 32~58%가 할인을 시작하고, 이 13일이 Epic 할인 시작의 약 72%입니다. 2026년에는 핼러윈 세일(10월 19일~11월 2일)도 포함됩니다. 세일은 매년 거의 같은 날짜에 열립니다(2월 9~10일, 3월 26~30일, 5월 14~15일, 7월 16~17일, 9월 3~4일, 12월 11~12일).
- **Epic의 V가 더 높은 이유는 세일 한 번의 크기가 아니라 월별 들쭉날쭉함 때문입니다.** 세일 한 번에 시작하는 게임 비율은 Steam(59~72%)이 더 큽니다. 하지만 Epic은 세일이 없는 달(1·4·6·8·11월)에 할인이 거의 시작되지 않아 월별 배율이 0.2~0.4배까지 떨어지고, 세일이 있는 달은 대부분 1.4~1.9배입니다. Steam은 큰 세일 사이에도 할인이 계속 시작돼서 월별로는 덜 극단적이고, 가을 세일이 9월과 10월로 갈라진 것도 Steam의 V를 낮춥니다.
- 의미: Steam과 Epic 같은 대형 스토어의 정기 세일은 **둘 다 뚜렷합니다.** V는 "한 해에 달마다 얼마나 들쭉날쭉한가"를 재는 값이라서 "누가 더 센 세일을 하는가"로 읽으면 안 됩니다.
- 주의: Epic은 이 데이터에서 게임이 340개입니다.

#### 6-2e) H2 결론 — 쏠림은 있고, 모든 스토어를 섞으면 희석된다

| 가설 | 판정 | 핵심 |
| --- | --- | --- |
| H2-a 모든 스토어 최저가 | **⚠️ 부분 일치** | 3·6·12월에 몰리지만 V=0.082로 실무 기준(0.09) 미만 |
| H2-b Steam만 | **✅ 일치** | V=0.128(신뢰구간 0.125~0.132), 정기 세일 시작일 8일에 Steam 할인 시작의 38%가 몰림 |

1. 할인은 3·6·12월(봄·여름·겨울 세일)에 몰리는 **시즌성이 있습니다.**
2. 모든 스토어의 최저가로 보면 쏠림이 **약하게** 나옵니다. 이는 쏠림이 없어서가 아니라 **스토어를 섞어서 희석된 것**으로 보입니다(할인 시작 건수 24% 감소, 1년의 79%가 할인 중).
3. **Steam만 보면 쏠림이 뚜렷합니다.** 가을 세일이 해마다 다른 달에 시작해서 9·10월도 함께 높습니다.
4. **단서**: "공식 스토어는 강하고 상시 할인 판매점은 약하다"로 일반화할 수는 없습니다. Steam과 Epic은 뚜렷하지만 GOG, Humble Store, Ubisoft, GamersGate는 약하고, 상시 할인 정도와 V의 스토어 간 상관은 약합니다(ρ=-0.44, p=0.20). 정확한 표현은 **"정기 세일에 게임이 한꺼번에 몰리는 스토어(Steam)에서는 뚜렷하고, 스토어를 섞으면 희석된다"** 입니다.
5. 서비스: Steam 가격 기준의 안내에는 공식 정기 세일 일정을 활용하고, 모든 스토어 최저가 기준의 월별 안내는 보조 정보로만 씁니다.

### 6-3) H3. 서비스의 "평균 간격" 규칙은 얼마나 정확한가? ★

**백테스트**: 각 게임에서 k번째 할인을 **앞의 k-1회만 보고** 예측해 실제와 비교합니다(과거를 가려 놓고 맞히기). 4가지 규칙을 같은 예측에 적용합니다.

| 규칙 | 예측 방법 |
| --- | --- |
| 직전 간격 1개 | 마지막 할인일 + 바로 전 간격 |
| **서비스 규칙** | 마지막 할인일 + 과거 간격의 **평균** |
| 중앙값 규칙 | 마지막 할인일 + 과거 간격의 **중앙값** |
| 전체 중앙값 | 마지막 할인일 + **모든 게임 간격의 중앙값** (예측 시점 **이전에 끝난 간격만** 사용 — 미래 데이터를 쓰지 않음) |

오차는 예측일과 실제 할인 시작일의 차이(일, 절댓값)입니다. 오차 분포가 한쪽으로 치우쳐 있어 평균 비교 대신 **Wilcoxon 부호순위검정**(같은 예측 대상에 대한 짝지은 자료, 단측)을 씁니다.

> **주의 — 예측들은 서로 독립이 아닙니다.** 한 게임에서 예측을 여러 번(평균 약 12번) 하는데, **같은 게임의 예측은 그 게임의 성격(규칙적인가 제멋대로인가)을 공유해서 같이 맞거나 같이 틀리기 쉽습니다.** 예를 들어 매달 꼬박꼬박 할인하는 게임은 6번 예측이 모두 잘 맞고, 제멋대로 할인하는 게임은 6번이 모두 크게 틀립니다. 이 12번을 서로 독립된 12개의 정보로 세면 p값이 실제보다 작게(더 확실한 것처럼) 나옵니다. 그래서 신뢰구간은 **같은 게임의 예측을 통째로 묶어서 게임 단위로 다시 뽑는 방식**(cluster bootstrap)으로 구했고, 판정은 p값보다 **효과 크기(몇 일 차이인가)** 를 우선합니다. 7장에서는 같은 게임의 예측을 **게임당 한 값으로 줄여서** 다시 검정해 이 문제를 아예 없앤 결과도 확인합니다.

In [ ]:
RULES = ['직전 간격 1개', '서비스 규칙(평균)', '중앙값 규칙', '전체 중앙값']

def backtest(ev, min_events=3):
    """각 게임의 k번째 할인을 앞의 k-1회로 예측했을 때 규칙별 오차(일). 전체 중앙값은 그 날짜 이전에 끝난 간격만 쓴다."""
    log = pd.DataFrame([(e['start'].iloc[i], (e['start'].iloc[i] - e['start'].iloc[i - 1]).days)
                        for e in ev.values() for i in range(1, len(e))], columns=['end', 'gap']).sort_values('end')
    end_arr, gap_arr = log['end'].to_numpy(), log['gap'].to_numpy()
    rows = []
    for gid, e in ev.items():
        if len(e) < min_events:
            continue
        st = e['start'].tolist()
        for k in range(2, len(st)):                                  # 앞의 k회(최소 2회)로 k번째 할인을 예측
            past, actual = st[:k], st[k]
            n = np.searchsorted(end_arr, np.datetime64(actual), side='left')     # 예측 날짜보다 먼저 끝난 간격 수
            if n < MIN_GLOBAL_GAPS:
                continue
            gaps = [(b - a).days for a, b in zip(past, past[1:])]
            miss = lambda days: abs((past[-1] + pd.Timedelta(days=float(days)) - actual).days)       # 오차(일)
            rows.append({'game_id': gid, RULES[0]: miss(gaps[-1]), RULES[1]: miss(round(np.mean(gaps))),
                         RULES[2]: miss(np.median(gaps)), RULES[3]: miss(np.median(gap_arr[:n]))})
    return pd.DataFrame(rows)

bt = backtest(events)
print(f"백테스트 예측 {len(bt):,}건 ({bt['game_id'].nunique():,}개 게임)")
tab = pd.DataFrame({'평균 오차(일)': bt[RULES].mean(), '중앙 오차(일)': bt[RULES].median(), '±14일 안에 맞힌 비율': (bt[RULES] <= 14).mean()})
display(tab.round(2))

In [ ]:
# 검정: 서비스 규칙의 오차가 더 작은가? (단측 Wilcoxon) + 게임 단위 부트스트랩 신뢰구간
SVC = RULES[1]
res3 = {}
for key, other in (('a', RULES[0]), ('b', RULES[3])):
    w = stats.wilcoxon(bt[SVC], bt[other], alternative='less')
    gain = float(bt[other].mean() - bt[SVC].mean())                    # +이면 서비스 규칙이 더 정확
    lo, hi = cluster_diff_ci(bt, SVC, other)                           # 서비스 - 기준 (음수 = 서비스가 더 정확)
    ok = bool(w.pvalue < ALPHA and gain >= PRACTICAL_DAYS and hi < 0)
    res3[key] = dict(other=other, p=float(w.pvalue), gain_days=gain, ci=[float(lo), float(hi)], ok=ok)
    print(f'H3-{key} 서비스 규칙 vs {other}: 평균 오차 {bt[SVC].mean():.1f}일 vs {bt[other].mean():.1f}일 (차이 {-gain:+.1f}일, '
          f'95% 신뢰구간 {lo:+.1f} ~ {hi:+.1f}일), p = {w.pvalue:.3g}')
    print('      판정:', verdict(w.pvalue, ok, direction_ok=gain > 0))
w_c = stats.wilcoxon(bt[RULES[2]], bt[SVC], alternative='less')
R['h3'] = dict(n_pred=len(bt), n_games=int(bt['game_id'].nunique()), mae={r: float(bt[r].mean()) for r in RULES},
               hit14={r: float((bt[r] <= 14).mean()) for r in RULES}, a=res3['a'], b=res3['b'], median_vs_mean_p=float(w_c.pvalue))
q = bt[RULES].quantile([.5, .8, .9]).T.round(0).astype(int)
q.columns = ['50%', '80%', '90%']
R['h3']['interval_days'] = q.to_dict()
print(f'\n중앙값 규칙 vs 평균 규칙(서비스): p = {w_c.pvalue:.3g}')

colors = [GRAY, RED, ORANGE, BLUE]
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.3))
mae = bt[RULES].mean()
ax[0].bar(RULES, mae, color=colors)
for i, vv in enumerate(mae):
    ax[0].text(i, vv + .3, f'{vv:.1f}일', ha='center', fontweight='bold')
ax[0].set(title='★ 규칙별 평균 오차 (낮을수록 정확)', ylabel='일', ylim=(0, mae.max() * 1.15)); ax[0].tick_params(axis='x', labelrotation=10)
tol = np.arange(0, 61)
for r, c in zip(RULES, colors):
    ax[1].plot(tol, [(bt[r] <= t).mean() * 100 for t in tol], label=r, color=c, lw=2)
ax[1].axvline(14, color='k', ls=':')
ax[1].set(title='예측일 앞뒤 N일 안에 실제 할인이 시작된 비율', xlabel='허용 오차 N(일)', ylabel='%'); ax[1].legend(fontsize=9)
savefig(fig, 'a1_5_h3.png'); plt.show()

print('예측일 ± N일 안에 실제 할인이 시작될 확률 (표의 값 = 필요한 N일)')
q

### 차트 해설
왼쪽은 규칙별 평균 오차(일), 오른쪽은 "예측일 앞뒤 N일 안에 실제 할인이 시작된 비율"입니다. 아래 표는 50%, 80%, 90% 확률로 맞히려면 예측일 앞뒤로 며칠을 잡아야 하는지입니다.

### 이 차트로 알 수 있는 것
- 왼쪽: 막대가 낮을수록 정확합니다. **서비스 규칙(빨강)이 가장 낮지 않으면** 게임별로 평균 간격을 구하는 수고가 정확도로 이어지지 않는다는 뜻입니다.
- 오른쪽: 곡선들이 거의 겹치면 **어느 규칙을 써도 정확도가 비슷**합니다. 점선(±14일)에서 약 절반만 맞힌다는 것을 읽을 수 있습니다.
- 아래 표: 날짜 하나로 단정하기보다 **구간으로 안내**할 때의 폭을 정하는 근거입니다.

### 의미 판단 (2026-11-26 데이터 기준)
- 서비스 규칙(평균 간격)은 **직전 간격 하나보다 4.3일 더 정확합니다**(23.2일 vs 27.4일, ✅). 그래서 완전히 틀린 규칙은 아닙니다.
- 하지만 게임 정보를 전혀 안 쓰고 **"모든 게임의 간격 중앙값"만 쓰는 가장 단순한 값(20.7일)보다는 2.4일 더 부정확합니다**(❌). "게임마다 평균 간격을 구하는" 수고가 정확도로 이어지지 않았습니다. 이유는 5장의 긴 꼬리(가끔 오래 쉬는 간격)가 평균을 끌어올리고, 게임당 간격이 몇 개 안 돼서 게임별 평균이 불안정하기 때문입니다. 평균 대신 중앙값을 써도 차이가 없습니다(p=0.12).
- **어떤 규칙을 써도 앞뒤 2주 안에 맞히는 비율은 절반 정도입니다.** 서비스 규칙은 ±14일에서 50%, ±29일에서 80%, ±45일에서 90%입니다. 날짜 하나로 단정하는 것은 위험하고, **구간으로 보여 주는 것이 정직합니다.**

## 7. 이 결과는 의미가 있는가?

p값만으로 결론을 내리지 않고 네 가지를 함께 봅니다.

1. **통계적 유의성**: p < 0.05
2. **효과 크기**: 실무 기준(H1 CV 0.2, H2 V 0.09, H3 2일)을 넘는가
3. **신뢰구간**: 게임 단위 95% 신뢰구간이 0(차이 없음)을 포함하지 않는가
4. **강건성**: 분석 조건을 바꿔도 결론이 유지되는가 — **할인으로 세는 최소 할인율**(10% → 20%, 30%), **할인이 많은 게임만** 쓰기, **공식 스토어(Steam)만** 쓰기

### 7-1) 분석 조건을 바꿔서 다시 계산

In [ ]:
# 일별 최저가: series_all(모든 스토어), series_steam(Steam만)은 앞에서 만들었다

# (이름, 일별 최저가, 할인으로 세는 최소 할인율, H1 최소 할인 횟수, H3 최소 할인 횟수)
SETTINGS = [('기본 (10% 이상, 모든 스토어)', series_all, 10, MIN_EVENTS, 3),
            ('20% 이상만 할인으로 셈', series_all, 20, MIN_EVENTS, 3),
            ('30% 이상만 할인으로 셈', series_all, 30, MIN_EVENTS, 3),
            ('할인 6회 이상인 게임만', series_all, 10, 6, 6),
            ('Steam만 (공식 스토어)', series_steam, 10, MIN_EVENTS, 3)]

rows = []
for name, ser, sm, h1_min, h3_min in SETTINGS:
    ev = make_events(ser, sm)
    g1 = h1_table(ev, h1_min)
    d1 = g1['cv'] - g1['cv_null']
    ob, ex = (m.sum(axis=0) for m in month_matrices(ser, ev))
    b3 = backtest(ev, h3_min)
    rows.append({'조건': name, 'H1 게임 수': len(g1), '무작위보다 규칙적(%)': (d1 < 0).mean() * 100, 'CV 중앙값': g1['cv'].median(),
                 'H2 Cramér V': cramers_v(ob, ex)[0], 'H3 예측 건수': len(b3),
                 '서비스 규칙 오차(일)': b3[SVC].mean(), '전체 중앙값 오차(일)': b3[RULES[3]].mean(),
                 '서비스-전체중앙값(일)': b3[SVC].mean() - b3[RULES[3]].mean()})
rob = pd.DataFrame(rows)
display(rob.round(2))
R['robustness'] = rob.round(4).to_dict('records')

fig, ax = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
y = np.arange(len(rob))[::-1]
ax[0].barh(y, rob['무작위보다 규칙적(%)'], color=BLUE); ax[0].axvline(50, color=ORANGE, ls='--', label='실무 기준 50%')
ax[0].set(title='H1 무작위보다 규칙적인 게임(%)', yticks=y, yticklabels=rob['조건']); ax[0].legend(fontsize=8)
ax[1].barh(y, rob['H2 Cramér V'], color=BLUE); ax[1].axvline(PRACTICAL_V, color=ORANGE, ls='--', label=f'실무 기준 {PRACTICAL_V}')
ax[1].set(title="H2 Cramér's V"); ax[1].legend(fontsize=8)
ax[2].barh(y, rob['서비스-전체중앙값(일)'], color=[RED if v_ > 0 else GREEN for v_ in rob['서비스-전체중앙값(일)']]); ax[2].axvline(0, color='k', lw=.8)
ax[2].set(title='H3-b 서비스 규칙 - 전체 중앙값 오차(일)\n(양수 = 서비스 규칙이 더 나쁨)')
savefig(fig, 'a1_6_robustness.png'); plt.show()

### 차트 해설
분석 조건을 바꿔 가며 같은 세 가지 결과(규칙성, 시즌성, 서비스 규칙 대비 전체 중앙값 오차)를 반복한 것입니다.

### 이 차트로 알 수 있는 것
- **왼쪽**: 모든 조건에서 막대가 50%(주황 점선)를 넘으면 "규칙적"이라는 결론이 조건에 좌우되지 않습니다.
- **가운데**: 모든 스토어 조건에서 V가 0.09(주황 점선)보다 작고 Steam만 넘으면, "모든 스토어는 약하고 Steam은 뚜렷하다"(6-2)가 조건 변경에도 유지된 것입니다.
- **오른쪽**: 빨간 막대는 서비스 규칙이 전체 중앙값보다 **더 나쁘다**는 뜻입니다. 모든 조건에서 빨강이면 "게임별 평균 간격이 도움이 되지 않는다"는 결론이 강건한 것입니다. 초록(서비스 규칙이 더 좋음)이 나오는 조건이 있으면 그 조건에서는 결론이 달라진다는 뜻이므로 8장에서 따로 짚습니다.

### 의미 판단 (2026-11-26 데이터 기준)
- **규칙성(H1)은 어떤 조건에서도 유지됩니다.** 다섯 조건 모두 85~99%의 게임이 무작위보다 규칙적입니다. 공식 스토어(Steam)만 보면 99.1%(CV 0.41)로 더 규칙적입니다.
- **"서비스 규칙이 전체 중앙값보다 못하다"(H3-b)도 모든 조건에서 유지됩니다**(1.0~2.4일 더 부정확). 다만 Steam만 보면 차이가 약 1일로 가장 작습니다.
- **시즌성(H2)은 스토어 기준에 따라 달라집니다.** 모든 스토어 최저가에서는 V가 0.07~0.08로 실무 기준(0.09)에 못 미치고, Steam만 보면 0.13으로 넘습니다. 할인 기준(10→30%)이나 게임 선택을 바꿔도 같은 패턴이라, 이 차이는 분석 조건이 아니라 **스토어를 섞은 효과**입니다(6-2에서 다룸).
- 종합: **규칙성과 "평균 규칙이 단순 기준보다 못함"은 조건과 무관하게 유지되고, 시즌성은 스토어에 따라 다릅니다.**

### 7-2) 게임당 1쌍으로 줄여서 다시 검정 (H3)
6-3의 검정은 예측 9,011번을 한 덩어리로 놓고 비교했습니다. 같은 게임의 예측은 같이 움직이므로(독립이 아님), **게임마다 평균 오차를 하나로 줄여서** 게임 719개를 1쌍씩으로 다시 검정합니다. 이렇게 하면 "같은 게임의 예측이 한 덩어리로 움직이는 문제"가 아예 없어집니다. 대신 비교 단위가 9,011쌍에서 719쌍으로 줄어듭니다.

In [ ]:
per_game = bt.groupby('game_id')[RULES].mean()                # 게임마다 평균 오차 (게임 1개 = 1쌍)
rows = []
for key, other in (('a', RULES[0]), ('b', RULES[3])):
    w = stats.wilcoxon(per_game[SVC], per_game[other], alternative='less')
    diff = float(per_game[SVC].mean() - per_game[other].mean())       # 음수 = 서비스 규칙이 더 정확
    diff_med = float((per_game[SVC] - per_game[other]).median())      # 평균은 일부 게임에 끌려가므로 중앙값도 함께 본다
    better = float((per_game[SVC] < per_game[other]).mean() * 100)    # 서비스 규칙이 더 정확한 게임의 비율
    rows.append({'비교': f'서비스 규칙 vs {other}', '게임 수': len(per_game), '서비스-기준 오차 차이 평균(일)': diff,
                 '서비스-기준 오차 차이 중앙값(일)': diff_med, '서비스 규칙이 더 정확한 게임(%)': better, 'Wilcoxon p (단측, 서비스가 더 작다)': w.pvalue})
    R['h3_game_level'] = {**R.get('h3_game_level', {}), key: dict(n_games=len(per_game), diff_days=diff, median_days=diff_med, share_better=better, p=float(w.pvalue))}
game_level = pd.DataFrame(rows)
display(game_level.round(3))

d_g = per_game[SVC] - per_game[RULES[3]]
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(d_g.clip(-40, 40), bins=40, color=BLUE)
ax.axvline(0, color='k', lw=1.5)
ax.axvline(d_g.median(), color=ORANGE, ls='--', lw=2, label=f'중앙값 {d_g.median():+.1f}일')
ax.set(title='게임별 오차 차이: 서비스 규칙 - 전체 중앙값 (양수 = 서비스 규칙이 더 나쁨)', xlabel='게임별 평균 오차 차이(일, ±40일 밖은 가장자리로 모음)', ylabel='게임 수'); ax.legend()
savefig(fig, 'a1_7_game_level.png'); plt.show()

### 차트 해설
게임마다 "서비스 규칙의 평균 오차 - 전체 중앙값의 평균 오차"를 구해 히스토그램으로 그린 것입니다. 0(검은 선)보다 **오른쪽**이면 그 게임에서는 서비스 규칙이 더 나쁩니다.

### 이 차트로 알 수 있는 것
- 막대가 0의 오른쪽에 더 많이 쌓여 있으면, **예측 건수를 세지 않고 게임을 단위로 봐도** 서비스 규칙이 전체 중앙값보다 나쁘다는 뜻입니다.
- 표의 "서비스 규칙이 더 정확한 게임(%)"이 50%보다 작으면 같은 결론입니다.
- **평균 차이는 일부 게임에 끌려 올라갑니다.** 예측이 몇 번 안 되는 게임은 오차가 아주 클 수 있어서, 게임마다 같은 비중으로 평균을 내면 예측 건수로 가중한 값(6-3장)보다 크게 나옵니다. 그래서 **중앙값과 "더 정확한 게임의 비율"** 을 함께 봅니다.

### 의미 판단 (2026-11-26 데이터 기준)
- **게임 단위로 줄여도 결론은 같습니다.** 719개 게임으로 비교하면 서비스 규칙은 직전 간격보다 정확하고(오차 차이 중앙값 -4.1일, **80%의 게임**에서 더 정확, ✅), 전체 중앙값보다는 부정확합니다(**서비스 규칙이 더 정확한 게임은 35%뿐**, 65%의 게임에서는 전체 중앙값이 더 정확, ❌).
- 따라서 "같은 게임의 예측이 한 덩어리로 움직여서 p값이 과장됐다"는 우려와 상관없이 6-3의 결론이 유지됩니다.
- **평균 차이(+8.4일)는 그대로 믿으면 안 되고, 중앙값(+1.1일)과 비율(35%)로 읽어야 합니다.** 예측이 몇 번 안 되는 일부 게임에서 오차가 아주 커서(30~46일) 평균이 끌려 올라갔습니다. 정확히는 "서비스 규칙이 전체 중앙값보다 훨씬 나쁘다"가 아니라 **"더 낫지 않고, 오히려 조금 나쁘다"** 입니다. (예측 단위의 평균 차이 2.4일과도 일관됩니다.)

### 7-3) 탐색: 이력이 긴 게임일수록 서비스 규칙이 나아지는가?
예측이 많이 나온 게임(= 할인 이력이 긴 게임)과 적은 게임에서 서비스 규칙의 상대적 정확도가 다른지 봅니다. **구간을 결과를 본 뒤에 나눈 탐색적 분석**이라 확정 결론이 아니라 가설을 만들기 위한 참고용입니다.

In [ ]:
per_game['예측 횟수'] = bt.groupby('game_id').size()
per_game['오차 차이'] = per_game[SVC] - per_game[RULES[3]]          # 양수 = 서비스 규칙이 전체 중앙값보다 나쁨
per_game['구간'] = pd.cut(per_game['예측 횟수'], [0, 5, 10, 20, 1000], labels=['1~5번', '6~10번', '11~20번', '21번 이상'])
by_len = per_game.groupby('구간', observed=True).agg(게임수=('오차 차이', 'size'), 평균차이=('오차 차이', 'mean'), 중앙차이=('오차 차이', 'median'),
                                                   서비스가더정확한게임=('오차 차이', lambda x: (x < 0).mean() * 100))
display(by_len.round(2))
R['h3_by_length'] = by_len.round(3).reset_index().astype({'구간': str}).to_dict('records')

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(by_len.index.astype(str), by_len['중앙차이'], color=[RED if v > 0 else GREEN for v in by_len['중앙차이']])
for i, (v, n_) in enumerate(zip(by_len['중앙차이'], by_len['게임수'])):
    ax.text(i, v + (.1 if v >= 0 else -.1), f'{v:+.1f}일\n(게임 {n_}개)', ha='center', va='bottom' if v >= 0 else 'top', fontsize=9)
ax.axhline(0, color='k', lw=.8)
ax.set(title='예측 횟수별 오차 차이 중앙값: 서비스 규칙 - 전체 중앙값 (양수 = 서비스 규칙이 더 나쁨)', xlabel='게임당 예측 횟수 (할인 이력이 길수록 많음)', ylabel='일')
ax.set_ylim(min(by_len['중앙차이'].min() - 1.2, -1.2), by_len['중앙차이'].max() + 1.8)
savefig(fig, 'a1_8_by_history_length.png'); plt.show()

### 차트 해설
게임을 "예측이 몇 번 나왔는가"로 나누고, 구간마다 "서비스 규칙의 오차 - 전체 중앙값의 오차"의 중앙값을 막대로 그렸습니다. 빨강은 서비스 규칙이 더 나쁜 구간, 초록은 더 나은 구간입니다.

### 이 차트로 알 수 있는 것
- 이력이 짧은 구간(왼쪽)에서 서비스 규칙이 전체 중앙값보다 더 나쁘고, 이력이 길어질수록(오른쪽) 격차가 줄어드는지 봅니다.
- 이력이 충분히 긴 구간에서 초록이 나오면, 게임별 평균 간격이 **이력이 충분히 쌓인 뒤에는** 도움이 된다는 뜻입니다.

### 의미 판단 (2026-11-26 데이터 기준)
- **이력이 짧은 게임(예측 1~5번, 134개)에서 서비스 규칙이 가장 약합니다.** 평균 차이 +38일, 중앙값도 +6.0일이고 서비스 규칙이 더 정확한 게임은 30%입니다. 이유는 간격이 한두 개뿐이라 "과거 간격의 평균"이 사실상 직전 간격 하나와 같아지기 때문입니다(할인 2~3회만으로도 예측해 버리는 규칙의 특성). 이런 게임은 **전체 중앙값이 더 안전**하다는 6-3장의 제안을 뒷받침합니다.
- **가장 큰 집단인 11~20번(318개)에서도 서비스 규칙이 더 정확한 게임은 21%뿐**이라 전체 중앙값이 우세합니다(중앙값 차이 +1.3일).
- **예측이 21번 이상인 게임(102개)에서만 서비스 규칙이 앞섭니다**(중앙값 -0.7일, 64%의 게임).
- **주의**: 이력이 길수록 매끄럽게 나아지는 추세가 아닙니다(6~10번 47%, 11~20번 21%, 21번 이상 64%로 들쭉날쭉). 마지막 구간은 게임이 102개뿐이고 구간을 결과를 본 뒤에 나눈 탐색이므로, "이력이 아주 길면 평균 규칙이 낫다"는 **확정 결론이 아니라 후속 검증이 필요한 가설**입니다.

## 8. 결론 — 가설 대비 결과와 인사이트·제안

아래 문장은 위 분석 결과를 **그대로 불러와** 만들어집니다. 판정이 "불일치"인 가설은 데이터가 지지하지 않는다고 발표에서 솔직하게 말하고, 제안은 일치한 부분만 근거로 삼습니다.

In [ ]:
h1r, h2r, h3r, h2s = R['h1'], R['h2'], R['h3'], R['h2_steam']
summary = pd.DataFrame([
    ['H1 할인은 규칙적이다', f"CV 중앙값 {h1r['cv_median']:.2f} (무작위 {h1r['cv_null_median']:.2f}), 규칙적인 게임 {h1r['share_regular']:.1%}", f"p={h1r['p']:.1e}", f"{h1r['ci'][0]:+.2f} ~ {h1r['ci'][1]:+.2f}", mark_r(h1r)],
    ['H2-a 특정 달에 몰린다 (모든 스토어)', f"V={h2r['v']:.3f}, 몰리는 달 {h2r['busy_months']}", f"p={h2r['p']:.1e}", f"V {h2r['ci'][0]:.3f} ~ {h2r['ci'][1]:.3f}", mark_r(h2r)],
    ['H2-b 특정 달에 몰린다 (Steam만)', f"V={h2s['v']:.3f}, 몰리는 달 {h2s['busy_months']}", f"p={h2s['p']:.1e}", f"V {h2s['ci'][0]:.3f} ~ {h2s['ci'][1]:.3f}", mark_r(h2s)],
    ['H3-a 서비스 규칙 > 직전 간격', f"{h3r['mae'][SVC]:.1f}일 vs {h3r['mae'][RULES[0]]:.1f}일", f"p={h3r['a']['p']:.1e}", f"{h3r['a']['ci'][0]:+.1f} ~ {h3r['a']['ci'][1]:+.1f}일", mark_r(h3r['a'])],
    ['H3-b 서비스 규칙 > 전체 중앙값', f"{h3r['mae'][SVC]:.1f}일 vs {h3r['mae'][RULES[3]]:.1f}일", f"p={h3r['b']['p']:.2f}", f"{h3r['b']['ci'][0]:+.1f} ~ {h3r['b']['ci'][1]:+.1f}일", mark_r(h3r['b'])],
], columns=['가설', '핵심 값', '검정', '95% 신뢰구간', '판정'])
summary

In [ ]:
rb = pd.DataFrame(R['robustness'])
svc_worse_everywhere = bool((rb['서비스-전체중앙값(일)'] > 0).all())
regular_everywhere = bool((rb['무작위보다 규칙적(%)'] >= 50).all())
small_season_everywhere = bool((rb['H2 Cramér V'] < PRACTICAL_V).all())
q = pd.DataFrame(h3r['interval_days'])
L = []
L.append('■ 가설 대비 결과')
L.append(f"  H1 규칙적: {mark_r(h1r)}. 게임 {h1r['n_games']}개 기준 실제 CV 중앙값 {h1r['cv_median']:.2f}, 무작위라면 {h1r['cv_null_median']:.2f}. 무작위보다 규칙적인 게임 {h1r['share_regular']:.1%}.")
L.append(f"  H2-a 시즌성(모든 스토어 최저가): {mark_r(h2r)}. 몰리는 달 {h2r['busy_months']}, Cramér's V = {h2r['v']:.3f}로 실무 기준({PRACTICAL_V}) {'이상' if h2r['v'] >= PRACTICAL_V else '미만(약함)'}.")
L.append(f"  H2-b 시즌성(Steam만): {mark_r(h2s)}. 몰리는 달 {h2s['busy_months']}, Cramér's V = {h2s['v']:.3f}로 실무 기준 {'이상(뚜렷함)' if h2s['v'] >= PRACTICAL_V else '미만'}. 정기 세일 시작일 {R['steam_big_days']['matched']}일에 Steam 할인 시작의 {R['steam_big_days']['share_of_starts']:.0f}%가 몰림.")
dl = R['h2_dilution']
L.append(f"  → 시즌 쏠림은 있지만 모든 스토어를 섞으면 희석됨: 할인 시작 건수가 Steam만 {dl['steam_starts']:,}건 → 모든 스토어 최저가 {dl['all_starts']:,}건으로 오히려 줄어듦(이미 어느 스토어에서 할인 중이면 새 시작으로 안 잡힘).")
L.append(f"  H3-a 서비스 규칙 > 직전 간격: {mark_r(h3r['a'])}. 평균 오차 {h3r['mae'][SVC]:.1f}일 vs {h3r['mae'][RULES[0]]:.1f}일.")
L.append(f"  H3-b 서비스 규칙 > 전체 중앙값: {mark_r(h3r['b'])}. 평균 오차 {h3r['mae'][SVC]:.1f}일 vs {h3r['mae'][RULES[3]]:.1f}일 (예측 시점 이전 데이터만 사용).")
gl = R['h3_game_level']['b']
L.append(f"  게임당 1쌍으로 줄여도(같은 게임의 예측을 하나로 묶음, {gl['n_games']}개 게임): 서비스 규칙이 전체 중앙값보다 정확한 게임 {gl['share_better']:.0f}%, 오차 차이 중앙값 {gl['median_days']:+.1f}일(평균 {gl['diff_days']:+.1f}일은 예측이 적은 일부 게임에 끌려 올라감), p = {gl['p']:.2f}.")
L.append(f"  조건을 바꿔도(5가지) 결론 유지: 규칙적={'예' if regular_everywhere else '아니오'}, 시즌 효과 작음={'예' if small_season_everywhere else '아니오(스토어에 따라 다름)'}, 서비스 규칙이 전체 중앙값보다 나쁨={'예' if svc_worse_everywhere else '아니오'}.")

L.append('\n■ 서비스 제안')
L.append(f"  - 다음 할인일을 날짜 하나가 아니라 구간으로 보여 줌: 예측일 ±{int(q.loc[SVC, '50%'])}일이면 50%, ±{int(q.loc[SVC, '80%'])}일이면 80% 확률로 실제 할인이 시작됨. 날짜 하나로 단정하면 절반만 ±14일 안에 맞음.")
if not h3r['b']['ok']:
    L.append(f"  - 게임별 평균 간격이 전체 중앙값({h3r['mae'][RULES[3]]:.1f}일 오차)보다 나을 이유를 찾지 못함 → 이력이 짧은 게임은 전체 중앙값을 기본값으로 쓰는 것을 검토.")
if h2s['ok']:
    L.append("  - Steam 가격 기준의 안내에는 공식 정기 세일(봄·여름·가을·겨울) 일정을 활용: 세일 시작일 하루에 게임의 6~7할이 한꺼번에 할인을 시작하므로 '곧 세일 시즌' 안내가 의미 있음. 일정은 해마다 바뀔 수 있어(가을 세일이 11월 말에서 앞당겨짐) Valve 공지를 따라야 함.")
if h2r['v'] < PRACTICAL_V:
    L.append("  - 모든 스토어 최저가 기준의 월별 시즌 안내는 보조 정보로만: 상시 할인에 가까운 판매점이 섞여 쏠림이 희석됨.")
L.append('  - 규칙 변경은 docs/contracts/timing_rules.md에 적고 backend 담당과 협의 (이 노트북은 근거 자료).')

L.append('\n■ 데이터·파이프라인 시사점')
L.append(f"  - 분석 가능한 게임 {R['n_games']}개 중 간격 분석 대상 {R['n_h1_games']}개. 가격 이력이 약 24개월이라 연간 패턴은 최대 2회분이므로, 이력이 더 쌓이면 같은 노트북으로 다시 검정할 수 있음.")
L.append('  - price_history는 가격이 바뀐 때만 쌓이므로 일별 최저가는 스토어별 마지막 가격을 이어 채워서 만든다(서비스 timing.py와 같은 방식).')
print('\n'.join(L))

(RESULT_DIR / 'summary.json').write_text(json.dumps(R, ensure_ascii=False, indent=2, default=float), encoding='utf-8')
print('\n요약 저장:', RESULT_DIR / 'summary.json')

### 수익 연결 (아이디어 · 구매 데이터가 없어 효과는 검증하지 못함)
팀이 정한 수익 모델은 **제휴 수수료 · 광고 배너 · 메일 제휴 추천**입니다. 이 분석과는 이렇게 이어 볼 수 있습니다.
* 다음 할인을 **구간**("○○ 전후")으로 알려 주는 안내는 사용자가 "곧 할인이 온다/아직 이르다"를 판단하게 돕고, 실제로 살 때 스토어 링크(제휴 수수료)로 이어질 수 있습니다. 스토어 링크 클릭은 이미 클릭스트림으로 쌓이고 있어 측정은 가능합니다.
* 단, 예측이 앞뒤 2주 오차가 있다는 점을 숨기면 신뢰를 잃으므로 구간으로 보여 주는 것이 전제입니다.

### 한계 (발표에서 먼저 밝힐 것)
| 한계 | 영향 | 대응 |
| --- | --- | --- |
| **"할인"을 모든 스토어 중 한 곳이라도 10% 이상이면 1회로 센다** | 상시 할인에 가까운 판매점이 섞여 간격이 짧게(중앙값 약 1개월) 나오고, 정기 세일의 시즌 쏠림도 희석됨 | 6-2에서 Steam만·스토어별로 따로 확인, 7장에서 20·30% 이상만 세기 등으로 결론이 유지되는지 확인 |
| 가격 이력이 약 24개월 | 연간 패턴은 최대 2회분, 시즌성 검정의 힘이 약함 | 이력이 쌓이면 다시 실행. Prophet 같은 연간 계절성 모델은 쓰지 않음 |
| **분석 대상이 "최근 2년에 4회 이상 할인한 게임"(약 68%)으로 한정** | 규칙성(H1) 결과는 무료이거나 할인을 거의 안 하는 게임에는 적용되지 않음 | 4장 "누가 분석에서 빠졌나" 표로 공개. 서비스 규칙도 할인 2회 미만은 예측하지 않음 |
| 한국어 리뷰가 많은 **인기작 위주** 1,000개 | 전체 게임으로 일반화할 수 없음 | `analysis/game_list/README.md`의 선정 기준 참고, 결과에 표본 수를 함께 표기 |
| 같은 세일에 많은 게임이 같이 들어감 | 게임 간 독립이 아니라 p값이 실제보다 작게 나옴 | 게임 단위 부트스트랩 신뢰구간, 판정은 효과 크기 우선 |
| 상관·패턴은 "함께 나타난다"이지 원인이 아님 | 할인 시기가 세일 이벤트 때문인지 퍼블리셔 전략인지 구분 못 함 | 해석은 "경향" 수준으로 |
| 과거를 가려 맞히는 백테스트 | 실제 서비스에서는 예측일이 이미 지났을 때의 처리(오늘 + 14일)가 따로 있음 | 이 노트북은 규칙 자체의 정확도만 검증 |